# Temporal validation, shift diagnosis and recalibration of return-to-work prediction models (PSWCI 2013–2024)

Reproduces **every table and figure** of the manuscript *"Temporal Validation, Shift Diagnosis and
Recalibration of Return-to-Work Prediction Models in Three National Workers' Compensation Cohorts"* (analysis v2:
retention and re-entry tasks, sample and population estimands, cohort-specific shift diagnosis, label-free and
lagged updating).

* Data: Korean Panel Study of Workers' Compensation Insurance (PSWCI, 산재보험패널조사), cohorts 1–3. The microdata are
  **public-use data** released free of charge by the Labor Welfare Research Institute of the Korea Workers' Compensation &
  Welfare Service (근로복지공단 근로복지연구원): anyone can download them from <https://www.pswci.or.kr/Data/Download.aspx>
  (button "산재보험패널 데이터 신청하기": online application with identity verification, then immediate download) after
  accepting the conditions of use (research purposes only, no transfer to third parties, citation of the source,
  notification of resulting publications to pswci@comwel.or.kr). Because of the no-transfer condition the raw files are
  not distributed with this notebook; place the three downloaded cohort folders under `DATA_ROOT`.
* The harmonized analysis file is rebuilt from the raw SPSS files with `pswci_harmonize.py` (no dependency on any
  pre-built file).
* Outputs are aggregated tables (CSV) and figures (PNG/PDF) only. Row-level predictions and fitted models are kept in a
  cache folder (`RTW_CACHE_DIR`) that must not be shared.
* Runtime: about 1 hour with 4 CPU cores (n_jobs = 4); longer on a 2-core Colab CPU runtime.


In [ ]:
# 1. Mount Google Drive (Colab) and set paths (all overridable through environment variables)
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
DATA_ROOT = os.environ.get("DATA_ROOT", "/content/drive/MyDrive/2026_fall semester/Dataset/industrial_pannel")
WORK = os.environ.get("RTW_WORK_DIR", "/content/rtw_work" if IN_COLAB else os.path.abspath("rtw_work"))
SRC_DIR = os.path.join(WORK, "src")
os.environ["DATA_ROOT"] = DATA_ROOT
os.environ.setdefault("RTW_OUT_DIR", os.path.join(WORK, "outputs"))
os.environ.setdefault("RTW_CACHE_DIR", os.path.join(WORK, "cache"))
os.environ["RTW_USE_PARQUET"] = "0"          # always rebuild the harmonized file from the raw SPSS data
os.environ.setdefault("JOBLIB_TEMP_FOLDER", os.path.join(WORK, "joblib_tmp"))
os.makedirs(SRC_DIR, exist_ok=True)
os.makedirs(os.environ["JOBLIB_TEMP_FOLDER"], exist_ok=True)
print("DATA_ROOT exists:", os.path.isdir(DATA_ROOT))
print("outputs ->", os.environ["RTW_OUT_DIR"])

In [ ]:
# 2. Pinned packages (installed only on Colab; local runs use the existing environment)
if IN_COLAB:
    %pip install -q numpy==2.2.0 pandas==2.3.3 scipy==1.17.1 scikit-learn==1.6.1 xgboost==3.2.0 lightgbm==4.6.0 catboost==1.2.10 interpret-core==0.7.8 shap==0.48.0 statsmodels==0.14.6 matplotlib==3.10.8 joblib==1.5.3 pyreadstat==1.3.3 pyarrow==24.0.0
    print("If numpy/scipy were upgraded, restart the runtime once and run all cells again.")

## 3. Analysis code (identical to `04_github_repo/src`)

In [ ]:
%%writefile {SRC_DIR}/pswci_harmonize.py
"""
pswci_harmonize.py
==================
Harmonize the three cohorts of the Korean Panel Study of Workers' Compensation
Insurance (PSWCI; 산재보험패널조사) into one long-format analysis file.

    Cohort 1: 2013-2017 (5 waves), workers whose compensated medical care ended in 2012
    Cohort 2: 2018-2022 (5 waves), care ended in 2017
    Cohort 3: 2023-2024 (2 waves), care ended in 2022

Usage (local or Google Colab):
    from pswci_harmonize import build_harmonized
    df = build_harmonized(DATA_ROOT)            # DATA_ROOT = folder that contains the cohort folders
    df.to_parquet("pswci_harmonized_long.parquet")

Coding traps handled here (each one was verified against the SPSS value labels):
  * Disability grade: cohort 1 codes "no disability" as 0, cohorts 2-3 as 15.
  * Self-rated health: cohorts 1-2 use 4 levels (1 = very poor ... 4 = very good);
    cohort 3 uses 5 levels in the opposite direction (1 = very healthy ... 5 = not at all).
  * Difficulty in daily activities: cohorts 1-2 code 1 = very difficult ... 5 = not at all;
    cohort 3 codes 1 = not at all ... 5 = very difficult.
  * Recovery after injury: cohort 1 has 3 levels (1 = fully recovered), cohorts 2-3 have
    5 levels (5 = fully recovered).
  * Age: cohort 1 has 5 groups (<=20s, 30s, ...), cohorts 2-3 have 4 groups (<=30s, ...).
  * Firm size: cohort 1 has 7 classes, cohorts 2-3 have 8 classes.
  * Rehabilitation-service flags: 1/2 in cohorts 1 and 3, 1/0 in cohort 2.
  * Smoking/drinking: G017/G018 in cohort 2 but G018/G019 in cohort 3 (G016/G017 = sleep).
  * Cohort 1 'accident' has no commuting category (commuting injuries cannot be identified
    and are coded 0); cohorts 2-3 code 3 = commuting (C2: pre-2018 employer-controlled
    commuting only).
  * "Long-lasting disability" block: item 1 = sensory impairment, item 2 = physical-activity
    limitation (walking/stairs/lifting). Use `phys_limitation` for physical limitation.
  * Cohort 1 wave-1 work ability refers to ~90 days after care closure (not the interview date).
Raw PSWCI data are public-use data (free download for anyone from
https://www.pswci.or.kr/Data/Download.aspx after accepting the conditions of use); because
the conditions prohibit transfer to third parties they are not redistributed with this code,
and this script only reads files the user downloaded.
"""
from __future__ import annotations

import os
import unicodedata

import numpy as np
import pandas as pd
import pyreadstat

# --------------------------------------------------------------------------------------
# File discovery (Google Drive on macOS stores Korean names in NFD; normalise before match)
# --------------------------------------------------------------------------------------

def _nfc(s: str) -> str:
    return unicodedata.normalize("NFC", s)


def find_file(root: str, must_contain: list[str], ext: str = ".sav") -> str:
    hits = []
    for dp, _, fns in os.walk(root):
        for fn in fns:
            p = os.path.join(dp, fn)
            q = _nfc(p)
            if q.lower().endswith(ext) and all(_nfc(s) in q for s in must_contain):
                hits.append(p)
    if not hits:
        raise FileNotFoundError(f"No {ext} file containing {must_contain} under {root}")
    return sorted(hits, key=len)[0]


def cohort_paths(root: str) -> dict[int, str]:
    return {
        1: find_file(root, ["1차 코호트", "LONG_DATA"]),
        2: find_file(root, ["2차 코호트", "pswci2_05_long"]),
        3: find_file(root, ["3차 코호트", "(4. 통합 데이터_long)"], ext=".sav"),
    }


# --------------------------------------------------------------------------------------
# Helpers
# --------------------------------------------------------------------------------------

def _valid(s: pd.Series, lo: float, hi: float) -> pd.Series:
    """Keep values inside [lo, hi]; refusal / don't-know codes (8, 9, 98, 99, ...) -> NaN."""
    s = pd.to_numeric(s, errors="coerce")
    return s.where((s >= lo) & (s <= hi))


def _yes(s: pd.Series, yes=1, no=(2,)) -> pd.Series:
    s = pd.to_numeric(s, errors="coerce")
    out = pd.Series(np.nan, index=s.index)
    out[s == yes] = 1.0
    out[s.isin(no)] = 0.0
    return out


def _read(path: str, cols: list[str]) -> pd.DataFrame:
    _, meta = pyreadstat.read_sav(path, metadataonly=True)
    have = [c for c in cols if c in meta.column_names]
    missing = sorted(set(cols) - set(have))
    if missing:
        print(f"[harmonize] note: {os.path.basename(path)} lacks {missing}")
    df, _ = pyreadstat.read_sav(path, usecols=have)
    for c in missing:
        df[c] = np.nan
    return df


# Column maps: harmonised name -> cohort-specific source column
MAP = {
    1: dict(p="p", wave="wave", sex="gender", age_raw="age", edu="edu", tenure="workperiod",
            accident="accident", care_dur="con", disa15="disa15",
            svc_any="service", svc_job="jobservice", svc_med="medservice", svc_soc="psyservice",
            emp3="emp03", emp6="emp06", wt_long="wt", wt_cross="wt",
            ex_days="Gb003003", ex_h="Gb003004", ex_m="Gb003005",
            work_ability="Bb007002", work_ability_postcare="Bb007001",
            srh="Gb001001", chronic="Gb001003", limitation="Gb003001", phys_lim="Gb003002", adl_help="Gb004001",
            adl_mental="Gb005001", adl_indoor="Gb005002", adl_outdoor="Gb005003", adl_work="Gb005004",
            life_sat="Gb008007", smoke="Gb010001", drink="Gb010007",
            recovery="Ga001002", pain_freq="Ga001003", pain_interf="Ga001004", preinj_health="Ga002001",
            firm_size="C002004", union="C002019", job_status="C001007", industry="C001003",
            marital="A003008", hh_size="I001001"),
    2: dict(p="p", wave="wave", sex="gender", age_raw="age4", edu="edu", tenure="workperiod14",
            accident="accident", injury_type="injurytype", injury_part="injurypart",
            care_dur="con16", disa15="disa15",
            svc_any="service", svc_job="jobservice", svc_soc="socservice",
            emp3="emp3", emp6="emp6", wt_long="wt1", wt_cross="wt2",
            ex_days="G015001", ex_h="G015002", ex_m="G015003",
            work_ability="B026001", work_ability_postcare="B025001",
            srh="G005001", chronic="G006001", limitation="G009001", phys_lim="G009002", adl_help="G010001",
            adl_mental="G012001", adl_indoor="G012002", adl_outdoor="G012003", adl_work="G012004",
            life_sat="G022007", smoke="G017001", drink="G018001",
            recovery="G001001", pain_freq="G002001", pain_interf="G003001", preinj_health="G004001",
            firm_size="C024001", union="C011001", job_status="C005001", industry="C002002",
            marital="A002002", hh_size="I001001"),
    3: dict(p="p", wave="wave", sex="gender", age_raw="age4", edu="edu", tenure="workperiod14",
            accident="accident", injury_type="injurytype", injury_part="injurypart",
            care_dur="con6", disa15="disa15",
            svc_any="service", svc_job="jobservice", svc_med="medservice", svc_soc="socservice",
            svc_owner="ownerservice",
            emp3="emp3", emp6="emp6", wt_long="wt1", wt_cross="wt2", strata="strata",
            ex_days="G016001", ex_h="G016002", ex_m="G016003",
            work_ability="B026001", work_ability_postcare="B025001",
            srh="G005001", chronic="G006001", limitation="G010001", phys_lim="G010002", adl_help="G012001",
            adl_mental="G014001", adl_indoor="G014002", adl_outdoor="G014003", adl_work="G014004",
            life_sat="G026007", smoke="G018001", drink="G019001",
            recovery="G001001", pain_freq="G002001", pain_interf="G003001", preinj_health="G004001",
            preinj_chronic="G004002",
            firm_size="C030001", union="C014001", job_status="C005001", industry="C002003",
            marital="A002002", hh_size="I001001",
            **{f"phq{i}": f"G0290{i:02d}" for i in range(1, 10)}),
}

# Variables measured (by design) only at wave 1 -> carried as baseline attributes
WAVE1_ONLY = ["recovery", "pain_freq", "pain_interf", "preinj_health", "preinj_chronic",
              "work_ability_postcare", "firm_size", "union", "job_status", "industry"]
# Administrative / sampling attributes fixed at baseline
FIXED = ["sex", "age_raw", "edu", "tenure", "accident", "injury_type", "injury_part",
         "care_dur", "disa15", "svc_any", "svc_job", "svc_med", "svc_soc", "svc_owner", "strata"]


def _load_cohort(root: str, cohort: int, path: str) -> pd.DataFrame:
    m = MAP[cohort]
    raw = _read(path, ["pid"] + list(m.values()))
    df = pd.DataFrame({"pid": raw["pid"].astype("int64")})
    for k, src in m.items():
        df[k] = raw[src].values
    df["cohort"] = cohort
    for k in set().union(*[set(v) for v in MAP.values()]) - set(m):
        df[k] = np.nan  # item not fielded in this cohort
    # broadcast wave-1 values of baseline-only items to every wave of the same person
    w1 = df[df["wave"] == 1].set_index("pid")
    for k in WAVE1_ONLY + FIXED:
        if k in df.columns:
            df[k] = df["pid"].map(w1[k])
    return df


def _harmonize(df: pd.DataFrame) -> pd.DataFrame:
    c = df["cohort"]
    out = pd.DataFrame({"cohort": c, "pid": df["pid"], "wave": df["wave"].astype(int)})
    out["uid"] = out["cohort"] * 100000 + out["pid"]
    out["year"] = np.select([c == 1, c == 2, c == 3], [2012, 2017, 2022]) + out["wave"]
    out["participated"] = (pd.to_numeric(df["p"], errors="coerce") == 1).astype(int)
    out["wt_long"] = pd.to_numeric(df["wt_long"], errors="coerce")
    out["wt_cross"] = pd.to_numeric(df["wt_cross"], errors="coerce")
    out["strata"] = df["strata"]

    # ---- demographics -----------------------------------------------------------------
    out["female"] = (pd.to_numeric(df["sex"], errors="coerce") == 2).astype(float)
    age = pd.to_numeric(df["age_raw"], errors="coerce")
    out["age_grp"] = np.where(c == 1, age.map({1: 1, 2: 1, 3: 2, 4: 3, 5: 4}), age)  # 1<=39 2=40s 3=50s 4>=60
    out["edu"] = _valid(df["edu"], 1, 5)                                               # 1 none ... 5 college+
    out["married"] = (pd.to_numeric(df["marital"], errors="coerce") == 2).astype(float).where(df["marital"].notna())
    out["hh_size"] = _valid(df["hh_size"], 1, 20)

    # ---- injury & compensation (baseline, administrative) ------------------------------
    acc = pd.to_numeric(df["accident"], errors="coerce")
    out["accident_type"] = acc                                    # 1 accident, 2 disease, 3 commuting (c2-3)
    out["disease"] = (acc == 2).astype(float)
    out["commuting"] = (acc == 3).astype(float)
    out["injury_type"] = pd.to_numeric(df["injury_type"], errors="coerce")
    part = pd.to_numeric(df["injury_part"], errors="coerce")
    out["injury_part"] = part
    region = pd.Series(np.nan, index=df.index, dtype=object)
    region[part.isin([5, 9])] = "spine"          # neck, low back
    region[part.isin([6, 7])] = "upper_limb"     # arm, hand/finger
    region[part.isin([10, 11, 12])] = "lower_limb"  # hip, leg, foot/toe
    region[part.isin([8, 19])] = "trunk"         # chest/back, abdomen
    region[part.isin([1, 2, 3, 4])] = "head_face"
    region[part == 13] = "multiple"
    region[part.isin([14, 15, 16, 17, 18, 20, 21])] = "systemic_other"
    out["injury_region"] = region
    out["care_dur"] = _valid(df["care_dur"], 1, 6)               # 1 <=3m ... 6 >2y
    out["tenure"] = _valid(df["tenure"], 1, 14)                   # 1 <1m ... 14 >=20y
    d15 = pd.to_numeric(df["disa15"], errors="coerce")
    grade = np.where(c == 1, d15.where(d15 >= 1), d15.where(d15 <= 14))
    out["disab_grade"] = grade                                    # 1 (most severe) ... 14; NaN = none
    out["disab_any"] = pd.Series(grade, index=df.index).notna().astype(float)
    g = pd.Series(grade, index=df.index)
    out["disab_cat"] = pd.cut(g, [0, 3, 7, 9, 12, 14], labels=[1, 2, 3, 4, 5]).astype(float).fillna(6)
    # 1: grade 1-3, 2: 4-7, 3: 8-9, 4: 10-12, 5: 13-14, 6: none

    for k in ["svc_any", "svc_job", "svc_med", "svc_soc", "svc_owner"]:
        if k not in df:
            out[k] = np.nan
            continue
        s = pd.to_numeric(df[k], errors="coerce")
        out[k] = np.where(c == 2, s.map({1: 1.0, 0: 0.0}), s.map({1: 1.0, 2: 0.0}))
    # cohort 2 has no medical-rehabilitation flag; cohort 1 "psyservice" = social/psychological

    # ---- job at time of injury (asked at wave 1) ----------------------------------------
    fs = pd.to_numeric(df["firm_size"], errors="coerce")
    out["firm_size"] = np.where(c == 1, fs.map({1: 1, 2: 2, 3: 3, 4: 4, 5: 5, 6: 6, 7: 6}),
                                fs.map({1: 1, 2: 2, 3: 3, 4: 3, 5: 4, 6: 5, 7: 6, 8: 6}))
    # 1 <5, 2 5-9, 3 10-29, 4 30-99, 5 100-299, 6 300+
    out["union"] = _yes(df["union"], 1, (2, 3))
    js = pd.to_numeric(df["job_status"], errors="coerce")
    out["job_status"] = js.map({1: 1, 2: 2, 3: 3, 4: 4, 5: 4})  # 1 regular 2 temporary 3 daily 4 self-employed
    ind = pd.to_numeric(df["industry"], errors="coerce")
    out["industry"] = ind.map(lambda x: {3: "manufacturing", 6: "construction", 8: "transport"}.get(x, "other")
                              if pd.notna(x) else np.nan)

    # ---- economic activity (all waves) --------------------------------------------------
    out["emp3"] = _valid(df["emp3"], 1, 3)       # 1 employed, 2 unemployed, 3 not in labour force
    out["emp6"] = _valid(df["emp6"], 1, 6)       # 1 original job, 2 re-employed, 3 self-employed, ...
    out["employed"] = (out["emp3"] == 1).astype(float).where(out["emp3"].notna())
    out["rtw_original"] = (out["emp6"] == 1).astype(float).where(out["emp6"].notna())

    # ---- exercise (all waves) -----------------------------------------------------------
    days = _valid(df["ex_days"], 0, 7)
    h = _valid(df["ex_h"], 0, 16)
    mnt = _valid(df["ex_m"], 0, 59)
    per_day = (h.fillna(0) * 60 + mnt.fillna(0)).where(h.notna() | mnt.notna())
    per_day = per_day.where(days > 0, 0.0).where(days.notna())
    out["ex_days"] = days
    out["ex_min_day"] = per_day
    out["ex_min_week"] = days * per_day
    out["inactive"] = (days == 0).astype(float).where(days.notna())
    out["regular_ex"] = ((days >= 3) & (per_day >= 30)).astype(float).where(days.notna())
    out["ex_150"] = (out["ex_min_week"] >= 150).astype(float).where(out["ex_min_week"].notna())

    # ---- function, health (all waves) ---------------------------------------------------
    out["work_ability"] = _valid(df["work_ability"], 0, 10)
    # Cohort 1, wave 1 asked work ability "about 90 days after care closure"; from wave 2 (and in
    # cohorts 2-3 at every wave) it refers to the interview date.
    out["work_ability_ref_interview"] = (~((c == 1) & (out["wave"] == 1))).astype(int)
    out["work_ability_postcare"] = _valid(df["work_ability_postcare"], 0, 10)
    srh = pd.to_numeric(df["srh"], errors="coerce")
    out["poor_health"] = np.where(c == 3, srh.map({1: 0, 2: 0, 3: 0, 4: 1, 5: 1}),
                                  srh.map({1: 1, 2: 1, 3: 0, 4: 0}))
    out["chronic"] = _yes(df["chronic"])
    # Item 1 of the "long-lasting disability or physical restriction" block = SENSORY impairment
    # (vision/hearing/speech); item 2 = substantial limitation in walking, climbing stairs,
    # lifting/carrying. `limitation` is kept for backward compatibility and equals item 1.
    out["limitation"] = _yes(df["limitation"])          # DEPRECATED name: sensory impairment
    out["sensory_impair"] = _yes(df["limitation"])
    out["phys_limitation"] = _yes(df["phys_lim"])
    out["adl_help"] = _valid(df["adl_help"], 1, 4)                    # higher = more help needed
    out["adl_help_need"] = (out["adl_help"] >= 3).astype(float).where(out["adl_help"].notna())
    for k in ["adl_mental", "adl_indoor", "adl_outdoor", "adl_work"]:
        v = _valid(df[k], 1, 5)
        out[k] = np.where(c == 3, v, 6 - v)                             # harmonised: 1 none ... 5 very difficult
        out[k + "_diff"] = (pd.Series(out[k], index=df.index) >= 4).astype(float).where(v.notna())
    ls = _valid(df["life_sat"], 1, 5)
    out["life_sat"] = 6 - ls                                            # higher = more satisfied
    out["smoker"] = _yes(df["smoke"])
    out["drinker"] = _yes(df["drink"], 1, (2, 3))  # cohort 1 has an unlabelled code 3 (19% at wave 1); questionnaire asks about quitting -> treated as not a current drinker

    # ---- wave-1-only items (broadcast as baseline attributes) ---------------------------
    rec = pd.to_numeric(df["recovery"], errors="coerce")
    out["fully_recovered"] = np.where(c == 1, rec.map({1: 1.0, 2: 0.0, 3: 0.0}),
                                      rec.map({1: 0.0, 2: 0.0, 3: 0.0, 4: 0.0, 5: 1.0}))
    out["pain_freq"] = _valid(df["pain_freq"], 1, 5)                   # 1 never ... 5 always
    out["pain_interf"] = _valid(df["pain_interf"], 1, 4)               # 1 not at all ... 4 always
    out["pain_interf_high"] = (out["pain_interf"] >= 3).astype(float).where(out["pain_interf"].notna())
    out["preinj_poor_health"] = (_valid(df["preinj_health"], 1, 5) >= 4).astype(float).where(
        _valid(df["preinj_health"], 1, 5).notna())
    out["preinj_chronic"] = _yes(df["preinj_chronic"])

    # ---- cohort-3-only extras -----------------------------------------------------------
    if all(f"phq{i}" in df for i in range(1, 10)):
        items = pd.concat([_valid(df[f"phq{i}"], 1, 4) - 1 for i in range(1, 10)], axis=1)
        out["phq9"] = items.sum(axis=1, min_count=9)
    else:
        out["phq9"] = np.nan

    # non-participants carry no wave-specific measurements
    tv = ["emp3", "emp6", "employed", "rtw_original", "ex_days", "ex_min_day", "ex_min_week",
          "inactive", "regular_ex", "ex_150", "work_ability", "poor_health", "chronic", "limitation",
          "sensory_impair", "phys_limitation", "adl_help", "adl_help_need", "adl_mental", "adl_indoor", "adl_outdoor", "adl_work",
          "adl_mental_diff", "adl_indoor_diff", "adl_outdoor_diff", "adl_work_diff", "life_sat",
          "smoker", "drinker", "married", "hh_size", "phq9"]
    out.loc[out["participated"] == 0, tv] = np.nan
    return out


def build_harmonized(root: str) -> pd.DataFrame:
    paths = cohort_paths(root)
    frames = [_harmonize(_load_cohort(root, k, p)) for k, p in paths.items()]
    df = pd.concat(frames, ignore_index=True)
    for col in df.columns:
        if df[col].dtype == object and col not in ("injury_region", "industry", "strata"):
            df[col] = pd.to_numeric(df[col], errors="coerce")
    return df.sort_values(["cohort", "pid", "wave"]).reset_index(drop=True)


if __name__ == "__main__":
    import sys
    root = sys.argv[1] if len(sys.argv) > 1 else "."
    out = sys.argv[2] if len(sys.argv) > 2 else "pswci_harmonized_long.parquet"
    d = build_harmonized(root)
    d.to_parquet(out, index=False)
    print(d.shape, "->", out)


In [ ]:
%%writefile {SRC_DIR}/rtw_config.py
"""Configuration: predictors, tasks, constants and paths (revision v2)."""
from __future__ import annotations

import os

SEED = 2026
N_JOBS = int(os.environ.get("RTW_N_JOBS", "4"))     # cap parallelism (shared machine)
B_BOOT = int(os.environ.get("RTW_B_BOOT", "1000"))  # bootstrap resamples for C3 metrics
B_SHIFT = int(os.environ.get("RTW_B_SHIFT", "200"))  # bootstrap resamples with re-estimated domain classifiers
R_UPDATE = int(os.environ.get("RTW_R_UPDATE", "50"))  # repetitions for updating experiments
B_SHAP = int(os.environ.get("RTW_B_SHAP", "50"))
FAST = os.environ.get("RTW_FAST", "0") == "1"       # debug mode: tiny searches (never used for reported results)

NUM_VARS = [
    "female", "edu", "married", "hh_size", "disease", "commuting", "care_dur", "tenure",
    "svc_any", "svc_job", "firm_size", "union", "ex_days", "regular_ex", "work_ability",
    "work_ability_postcare", "chronic", "phys_limitation", "sensory_impair", "adl_help", "adl_mental", "adl_indoor",
    "adl_outdoor", "adl_work", "pain_freq", "pain_interf", "smoker", "drinker",
]
# ordinal scores that ridge-OH encodes as categories (levels fixed a priori)
ORDINAL_LEVELS = {
    "edu": [1, 2, 3, 4, 5], "hh_size": [1, 2, 3, 4, 5, 6], "care_dur": [1, 2, 3, 4, 5, 6],
    "tenure": list(range(1, 15)), "firm_size": [1, 2, 3, 4, 5, 6], "ex_days": list(range(0, 8)),
    "work_ability": list(range(0, 11)), "work_ability_postcare": list(range(0, 11)), "adl_help": [1, 2, 3, 4],
    "adl_mental": [1, 2, 3, 4, 5], "adl_indoor": [1, 2, 3, 4, 5], "adl_outdoor": [1, 2, 3, 4, 5],
    "adl_work": [1, 2, 3, 4, 5], "pain_freq": [1, 2, 3, 4, 5], "pain_interf": [1, 2, 3, 4],
}
NOM_LEVELS = {
    "age_grp": [1, 2, 3, 4],
    "job_status": [1, 2, 3, 4],
    "industry": ["construction", "manufacturing", "other", "transport"],
    "disab_cat": [1, 2, 3, 4, 5, 6],
}
EMP_VARS = ["employed", "emp6"]
PREDICTORS = NUM_VARS + list(NOM_LEVELS) + EMP_VARS          # 34 original variables
RESTRICTED_DROP = ["svc_any", "drinker", "adl_mental", "adl_indoor", "adl_outdoor", "adl_work",
                   "sensory_impair", "phys_limitation"]
RESTRICTED_DROP_B = RESTRICTED_DROP + ["svc_job", "tenure"]   # v2: + definition change (svc_job) and C1 tenure anomaly
SMALL_VARS = ["age_grp", "disab_cat", "employed", "work_ability", "pain_interf"]   # parsimonious LR

VAR_LABELS = {
    "female": "Female", "age_grp": "Age group", "edu": "Education", "married": "Married",
    "hh_size": "Household size", "disease": "Occupational disease", "commuting": "Commuting injury",
    "care_dur": "Duration of medical care", "tenure": "Job tenure at injury",
    "disab_cat": "Disability grade category", "svc_any": "Any rehabilitation service",
    "svc_job": "Vocational/employment-support service", "firm_size": "Firm size", "union": "Union at workplace",
    "job_status": "Job status at injury", "industry": "Industry", "employed": "Employed (wave 1)",
    "emp6": "Employment type (wave 1)", "ex_days": "Exercise days/week", "regular_ex": "Regular exercise",
    "work_ability": "Work ability (0-10)", "work_ability_postcare": "Work ability after care (recall)",
    "chronic": "Chronic disease", "phys_limitation": "Physical limitation",
    "sensory_impair": "Sensory impairment", "adl_help": "Need for help in ADL",
    "adl_mental": "Difficulty: mental activities", "adl_indoor": "Difficulty: indoor activities",
    "adl_outdoor": "Difficulty: outdoor activities", "adl_work": "Difficulty: work activities",
    "pain_freq": "Pain frequency", "pain_interf": "Pain interference", "smoker": "Current smoker",
    "drinker": "Current drinker",
}

# tasks: A1 retention (employed w1), A2 re-entry (not employed w1), P pooled (secondary)
TASKS = {"A1": "Job retention (employed at wave 1)", "A2": "Re-entry (not employed at wave 1)",
         "P": "Pooled employment (secondary)"}
# undesired event per task (target of an RTW-support flag): y = employed at wave 2, event = 1 - y
EVENT_LABEL = {"A1": "job loss", "A2": "no re-entry", "P": "non-employment"}
FAIR_THRESHOLD = {"A1": 0.10, "A2": 0.60, "P": 0.30}     # development prevalence of the undesired event (rounded)
DCA_THRESHOLDS = {"A1": [0.05, 0.10, 0.20, 0.30], "A2": [0.40, 0.50, 0.60, 0.70], "P": [0.10, 0.20, 0.30, 0.40]}

CANDIDATES = ["LR", "RidgeOH", "RF", "XGB", "LGBM", "CatBoost", "EBM"]
BASELINES = {"A1": ["Null", "LR-small"], "A2": ["Null", "LR-small"], "P": ["Null", "Status", "LR-small"]}
MODEL_LABEL = {"Null": "Null (development prevalence)", "Status": "Status-only LR", "LR-small": "Parsimonious LR",
               "LR": "LR (linear scores)", "RidgeOH": "Ridge LR (one-hot)", "RF": "Random forest", "XGB": "XGBoost",
               "LGBM": "LightGBM", "CatBoost": "CatBoost", "EBM": "EBM"}


def paths():
    data_root = os.environ.get(
        "DATA_ROOT", "/content/drive/MyDrive/2026_fall semester/Dataset/industrial_pannel")
    out_dir = os.environ.get("RTW_OUT_DIR", os.path.join(os.getcwd(), "rtw_outputs"))
    cache_dir = os.environ.get("RTW_CACHE_DIR", os.path.join(os.getcwd(), "rtw_cache"))
    for d in (out_dir, cache_dir, os.path.join(out_dir, "tables"), os.path.join(out_dir, "figures")):
        os.makedirs(d, exist_ok=True)
    return data_root, out_dir, cache_dir


In [ ]:
%%writefile {SRC_DIR}/rtw_data.py
"""Analytic data set and design matrices (wave-1 predictors -> wave-2 employment), revision v2."""
from __future__ import annotations

import os

import numpy as np
import pandas as pd

from rtw_config import NOM_LEVELS, NUM_VARS, ORDINAL_LEVELS, PREDICTORS, SMALL_VARS


def load_long(data_root: str, use_parquet: bool | None = None) -> pd.DataFrame:
    pq = os.path.join(data_root, "00_derived_data", "pswci_harmonized_long.parquet")
    if use_parquet is None:
        use_parquet = os.environ.get("RTW_USE_PARQUET", "0") == "1"
    if use_parquet and os.path.exists(pq):
        return pd.read_parquet(pq)
    from pswci_harmonize import build_harmonized
    return build_harmonized(data_root)


def build_analytic(long: pd.DataFrame) -> pd.DataFrame:
    """One row per wave-1 participant: wave-1 predictors, wave-2 participation/outcome, weights, strata."""
    w1 = long[(long["wave"] == 1) & (long["participated"] == 1)].set_index("uid")
    w2 = long[long["wave"] == 2].set_index("uid")
    a = w1[["cohort", "pid", "wt_long", "accident_type", "disab_grade", "strata"] + PREDICTORS].copy()
    a = a.rename(columns={"wt_long": "wt_long_w1"})
    a["part2"] = w2["participated"].reindex(a.index).fillna(0).astype(int)
    a["y"] = w2["employed"].reindex(a.index)               # employed at wave 2 (emp3 == 1)
    a["wt_w2"] = w2["wt_long"].reindex(a.index)            # longitudinal weight at wave 2
    a["hh_size"] = a["hh_size"].clip(upper=6)
    a["sex_grp"] = np.where(a["female"] == 1, "Female", "Male")
    a["age_lab"] = a["age_grp"].map({1: "<=39", 2: "40-49", 3: "50-59", 4: ">=60"})
    a["disab_grp"] = np.select([a["disab_cat"] == 6, a["disab_cat"].isin([3, 4, 5]), a["disab_cat"].isin([1, 2])],
                               ["No grade", "Mild (8-14)", "Severe (1-7)"], default="NA")
    a["acc_grp"] = np.where(a["disease"] == 1, "Disease", "Injury")
    a["job_grp"] = np.where(a["job_status"] == 1, "Regular", "Non-regular")
    return a


def task_frame(a: pd.DataFrame, task: str) -> pd.DataFrame:
    """Eligible rows: wave-2 respondents; A1 employed at wave 1; A2 not employed at wave 1; P all."""
    d = a[(a["part2"] == 1) & a["y"].notna()].copy()
    if task in ("A1", "B1"):
        d = d[d["employed"] == 1].copy()
    elif task in ("A2", "B"):
        d = d[d["employed"] == 0].copy()
    d["y"] = d["y"].astype(int)
    return d


def design_matrix(d: pd.DataFrame, task: str, variables: list[str] | None = None, ordinal_onehot: bool = False):
    """Return (X, groups): design columns and the original variable of each column.

    Levels of all one-hot encodings are fixed a priori (no data-driven learning; no leakage).
    ordinal_onehot=True encodes the ordinal scores of ORDINAL_LEVELS as categories (ridge-OH model).
    Employment status at wave 1 is encoded as employed + nested emp6 indicators; constant columns of a task dropped.
    """
    variables = list(variables or PREDICTORS)
    cols, groups = {}, {}
    for v in NUM_VARS:
        if v not in variables:
            continue
        if ordinal_onehot and v in ORDINAL_LEVELS:
            levels = ORDINAL_LEVELS[v]
            for lev in levels[1:]:
                name = f"{v}={lev}"
                cols[name] = (d[v] == lev).astype(float).where(d[v].notna())
                groups[name] = v
        else:
            cols[v] = d[v].astype(float)
            groups[v] = v
    for v, levels in NOM_LEVELS.items():
        if v in variables:
            for lev in levels[1:]:
                name = f"{v}={lev}"
                cols[name] = (d[v] == lev).astype(float).where(d[v].notna())
                groups[name] = v
    e6 = d["emp6"]
    if task in ("P",):
        if "employed" in variables:
            cols["employed"] = d["employed"].astype(float)
            groups["employed"] = "employed"
    if "emp6" in variables:
        if task in ("P", "A1"):
            cols["emp6=new_employer"] = (e6 == 2).astype(float)
            cols["emp6=self_employed"] = e6.isin([3, 4]).astype(float)
            groups["emp6=new_employer"] = groups["emp6=self_employed"] = "emp6"
        if task in ("P", "A2"):
            cols["emp6=job_seeking"] = (e6 == 5).astype(float)
            groups["emp6=job_seeking"] = "emp6"
    X = pd.DataFrame(cols, index=d.index)
    return X, groups


def small_vars(task: str) -> list[str]:
    return [v for v in SMALL_VARS if not (task != "P" and v == "employed")]


def status_columns(X: pd.DataFrame) -> list[str]:
    return [c for c in X.columns if c == "employed" or c.startswith("emp6=")]


In [ ]:
%%writefile {SRC_DIR}/rtw_metrics.py
"""Discrimination, calibration, overall and utility metrics with (design-based) bootstrap CIs (revision v2).

Predictions p are P(employed at wave 2). Sample weights w (survey weights, inverse probability of response weights
(IPRW), importance weights) are optional.
"""
from __future__ import annotations

import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score

EPS = 1e-6


def logit(p):
    p = np.clip(np.asarray(p, float), EPS, 1 - EPS)
    return np.log(p / (1 - p))


def expit(z):
    return 1.0 / (1.0 + np.exp(-np.asarray(z, float)))


def _w(y, w):
    return np.ones(len(y)) if w is None else np.asarray(w, float)


def newton_logistic(X, y, w=None, offset=None, n_iter=60, tol=1e-9):
    """Weighted logistic regression by Newton-Raphson. Returns (beta, converged)."""
    y = np.asarray(y, float)
    w = _w(y, w)
    off = 0.0 if offset is None else np.asarray(offset, float)
    beta = np.zeros(X.shape[1])
    for _ in range(n_iter):
        mu = expit(X @ beta + off)
        g = X.T @ (w * (y - mu))
        H = (X * (w * mu * (1 - mu))[:, None]).T @ X
        try:
            step = np.linalg.solve(H + 1e-10 * np.eye(len(beta)), g)
        except np.linalg.LinAlgError:
            return beta, False
        beta = beta + step
        if not np.all(np.isfinite(beta)):
            return beta, False
        if np.max(np.abs(step)) < tol:
            return beta, True
    return beta, False


def logistic_recal(y, lp, w=None, slope=True):
    lp = np.asarray(lp, float)
    if slope:
        X = np.column_stack([np.ones_like(lp), lp])
        beta, ok = newton_logistic(X, y, w)
        return beta, ok
    beta, ok = newton_logistic(np.ones((len(lp), 1)), y, w, offset=lp)
    return beta, ok


def citl(y, p, w=None):
    b, ok = logistic_recal(y, logit(p), w, slope=False)
    return float(b[0]) if ok else np.nan


def cal_slope(y, p, w=None):
    """Calibration slope; undefined (NaN) for (near-)constant predictions, non-convergence or |slope| > 50."""
    lp = logit(p)
    if np.std(lp) < 1e-3:
        return np.nan
    b, ok = logistic_recal(y, lp, w, slope=True)
    if not ok or abs(b[1]) > 50:
        return np.nan
    return float(b[1])


def brier(y, p, w=None):
    w = _w(y, w)
    return float(np.sum(w * (np.asarray(p) - np.asarray(y)) ** 2) / np.sum(w))


def scaled_brier(y, p, w=None):
    w = _w(y, w)
    ybar = np.sum(w * y) / np.sum(w)
    return 1.0 - brier(y, p, w) / (ybar * (1 - ybar))


def logloss(y, p, w=None):
    w = _w(y, w)
    p = np.clip(np.asarray(p, float), EPS, 1 - EPS)
    y = np.asarray(y, float)
    return float(-np.sum(w * (y * np.log(p) + (1 - y) * np.log(1 - p))) / np.sum(w))


def ece(y, p, w=None, bins=10):
    y = np.asarray(y, float)
    p = np.asarray(p, float)
    w = _w(y, w)
    order = np.argsort(p, kind="mergesort")
    tot = np.sum(w)
    e = 0.0
    for idx in np.array_split(order, bins):
        if len(idx):
            wb = w[idx]
            e += wb.sum() / tot * abs(np.sum(wb * y[idx]) / wb.sum() - np.sum(wb * p[idx]) / wb.sum())
    return float(e)


def _rcs_basis(x, knots):
    """Restricted cubic spline basis (Harrell): x plus len(knots)-2 non-linear terms."""
    k = np.asarray(knots, float)
    t = len(k)
    cols = [x]
    norm = (k[-1] - k[0]) ** 2
    for j in range(t - 2):
        term = (np.maximum(x - k[j], 0) ** 3
                - np.maximum(x - k[t - 2], 0) ** 3 * (k[t - 1] - k[j]) / (k[t - 1] - k[t - 2])
                + np.maximum(x - k[t - 1], 0) ** 3 * (k[t - 2] - k[j]) / (k[t - 1] - k[t - 2]))
        cols.append(term / norm)
    return np.column_stack(cols)


def calibration_curve_fit(y, p, w=None):
    """Flexible calibration: logistic regression of y on an RCS of logit(p) (4 knots). Returns a function."""
    lp = logit(p)
    if np.std(lp) < 1e-3:
        return None
    knots = np.quantile(lp, [0.05, 0.35, 0.65, 0.95])
    if len(np.unique(knots)) < 4:
        knots = np.quantile(lp, [0.05, 0.95])
        B = lambda z: np.column_stack([np.ones_like(z), z])  # noqa: E731
    else:
        B = lambda z: np.column_stack([np.ones_like(z), _rcs_basis(z, knots)])  # noqa: E731
    beta, ok = newton_logistic(B(lp), y, w)
    if not ok:
        return None
    return lambda q: expit(B(logit(q)) @ beta)


def ici(y, p, w=None):
    """Integrated calibration index and E50/E90 (Austin & Steyerberg) from the spline calibration curve."""
    f = calibration_curve_fit(y, p, w)
    if f is None:
        return np.nan, np.nan, np.nan
    d = np.abs(f(p) - np.asarray(p))
    wv = _w(y, w)
    order = np.argsort(d)
    cw = np.cumsum(wv[order]) / wv.sum()
    e50 = d[order][np.searchsorted(cw, 0.5)]
    e90 = d[order][min(np.searchsorted(cw, 0.9), len(d) - 1)]
    return float(np.sum(wv * d) / wv.sum()), float(e50), float(e90)


def auroc(y, p, w=None):
    if len(np.unique(y)) < 2 or np.ptp(np.asarray(p, float)) < 1e-12:
        return np.nan if len(np.unique(y)) < 2 else 0.5
    return float(roc_auc_score(y, p, sample_weight=w))


def minority_event(task):
    """AUPRC is computed for the minority class: A1 job loss, A2 re-entry, P non-employment."""
    return "re-entry" if task in ("A2", "B") else "not employed"


def auprc(y, p, task, w=None):
    y = np.asarray(y)
    p = np.asarray(p)
    if task in ("A2", "B"):
        return float(average_precision_score(y, p, sample_weight=w))
    return float(average_precision_score(1 - y, 1 - p, sample_weight=w))


def auprc_noskill(y, task, w=None):
    wv = _w(y, w)
    prev = np.sum(wv * y) / wv.sum()
    return float(prev if task in ("A2", "B") else 1 - prev)


METRICS = ("AUROC", "AUPRC", "Brier", "sBrier", "LogLoss", "CITL", "Slope", "ICI", "E50", "E90", "ECE")


def all_metrics(y, p, task, w=None, which=METRICS) -> dict:
    y = np.asarray(y, int)
    p = np.clip(np.asarray(p, float), EPS, 1 - EPS)
    wv = _w(y, w)
    out = {}
    if any(k in which for k in ("ICI", "E50", "E90")):
        out["ICI"], out["E50"], out["E90"] = ici(y, p, w)
    for k in which:
        if k == "AUROC":
            out[k] = auroc(y, p, w)
        elif k == "AUPRC":
            out[k] = auprc(y, p, task, w)
        elif k == "Brier":
            out[k] = brier(y, p, w)
        elif k == "sBrier":
            out[k] = scaled_brier(y, p, w)
        elif k == "LogLoss":
            out[k] = logloss(y, p, w)
        elif k == "CITL":
            out[k] = citl(y, p, w)
        elif k == "Slope":
            out[k] = cal_slope(y, p, w)
        elif k == "ECE":
            out[k] = ece(y, p, w)
    out["MeanPred"] = float(np.sum(wv * p) / wv.sum())
    out["Observed"] = float(np.sum(wv * y) / wv.sum())
    out["NoSkillAUPRC"] = auprc_noskill(y, task, w)
    return {k: out[k] for k in list(which) + ["MeanPred", "Observed", "NoSkillAUPRC"]}


def boot_indices(n, B, seed, strata=None):
    """Bootstrap index sets; resampling within strata when strata is given (design-based)."""
    rng = np.random.default_rng(seed)
    if strata is None:
        return [rng.integers(0, n, n) for _ in range(B)]
    strata = np.asarray(strata)
    groups = [np.where(strata == s)[0] for s in pd.unique(strata)]
    out = []
    for _ in range(B):
        out.append(np.concatenate([g[rng.integers(0, len(g), len(g))] for g in groups]))
    return out


def evaluate(y, preds: dict, task, B=1000, seed=2026, w=None, strata=None, refs=(), which=METRICS,
             label="unweighted"):
    """Point estimates and percentile CIs for every model; paired differences vs each model in refs."""
    y = np.asarray(y, int)
    idxs = boot_indices(len(y), B, seed, strata)
    point = {m: all_metrics(y, p, task, w, which) for m, p in preds.items()}
    samples = {m: {k: [] for k in which} for m in preds}
    dsamp = {}
    for idx in idxs:
        yb = y[idx]
        wb = None if w is None else np.asarray(w)[idx]
        res = {}
        for m, p in preds.items():
            res[m] = all_metrics(yb, np.asarray(p)[idx], task, wb, which)
            for k in which:
                samples[m][k].append(res[m][k])
        for ref in refs:
            if ref not in res:
                continue
            for m in preds:
                if m != ref:
                    for k in ("AUROC", "Brier", "LogLoss", "ICI"):
                        if k in which:
                            dsamp.setdefault((m, ref, k), []).append(res[m][k] - res[ref][k])
    rows = []
    for m in preds:
        for k in which:
            s = np.asarray(samples[m][k], float)
            ok = np.isfinite(s)
            rows.append({"model": m, "metric": k, "estimate": point[m][k],
                         "lo": np.percentile(s[ok], 2.5) if ok.sum() else np.nan,
                         "hi": np.percentile(s[ok], 97.5) if ok.sum() else np.nan,
                         "n_undefined": int((~ok).sum()), "weighting": label})
        for k in ("MeanPred", "Observed", "NoSkillAUPRC"):
            rows.append({"model": m, "metric": k, "estimate": point[m][k], "lo": np.nan, "hi": np.nan,
                         "n_undefined": 0, "weighting": label})
    drows = []
    for (m, ref, k), s in dsamp.items():
        s = np.asarray(s, float)
        drows.append({"model": m, "reference": ref, "metric": k, "estimate": point[m][k] - point[ref][k],
                      "lo": np.nanpercentile(s, 2.5), "hi": np.nanpercentile(s, 97.5),
                      "p_two_sided": float(min(1.0, 2 * min(np.mean(s <= 0), np.mean(s >= 0)))),
                      "weighting": label})
    return pd.DataFrame(rows), pd.DataFrame(drows)


# ---- decision curves -------------------------------------------------------------------------

def net_benefit(event, flag, pt, w=None):
    event = np.asarray(event, int)
    flag = np.asarray(flag, int)
    w = _w(event, w)
    n = np.sum(w)
    tp = np.sum(w * (flag == 1) * (event == 1)) / n
    fp = np.sum(w * (flag == 1) * (event == 0)) / n
    return tp - fp * pt / (1 - pt)


def dca_curves(y, preds: dict, thresholds, rule_flags: dict | None = None, w=None):
    """Net benefit of flagging for the undesired event (1 - y) with risk = 1 - p."""
    event = 1 - np.asarray(y, int)
    rows = []
    for pt in thresholds:
        rows.append({"threshold": pt, "strategy": "Flag all", "net_benefit": net_benefit(event, np.ones_like(event), pt, w)})
        rows.append({"threshold": pt, "strategy": "Flag none", "net_benefit": 0.0})
        for r, fl in (rule_flags or {}).items():
            rows.append({"threshold": pt, "strategy": r, "net_benefit": net_benefit(event, fl, pt, w)})
        for m, p in preds.items():
            rows.append({"threshold": pt, "strategy": m,
                         "net_benefit": net_benefit(event, ((1 - np.asarray(p)) >= pt).astype(int), pt, w)})
    return pd.DataFrame(rows)


def dca_vs_best_default(y, p, thresholds, rule_flags: dict | None = None, w=None, strata=None, B=1000, seed=2026):
    """NB(model) - max(NB(flag all), 0, NB(rules)) with bootstrap CIs (design-based if strata)."""
    y = np.asarray(y, int)
    event = 1 - y
    p = np.asarray(p)
    rule_flags = {k: np.asarray(v) for k, v in (rule_flags or {}).items()}

    def delta(idx, pt):
        ev = event[idx]
        wb = None if w is None else np.asarray(w)[idx]
        nb_m = net_benefit(ev, ((1 - p[idx]) >= pt).astype(int), pt, wb)
        defaults = [0.0, net_benefit(ev, np.ones_like(ev), pt, wb)] + [net_benefit(ev, f[idx], pt, wb)
                                                                       for f in rule_flags.values()]
        return nb_m - max(defaults), nb_m
    full = np.arange(len(y))
    idxs = boot_indices(len(y), B, seed, strata)
    rows = []
    for pt in thresholds:
        est, nbm = delta(full, pt)
        s = np.array([delta(i, pt)[0] for i in idxs])
        rows.append({"threshold": pt, "nb_model": nbm, "delta_vs_best_default": est,
                     "lo": np.percentile(s, 2.5), "hi": np.percentile(s, 97.5)})
    return pd.DataFrame(rows)


def calibration_band(y, p, w=None, B=200, seed=2026, strata=None, grid=None):
    """Spline calibration curve on a grid with pointwise percentile bands."""
    p = np.asarray(p, float)
    grid = np.linspace(np.quantile(p, 0.02), np.quantile(p, 0.98), 40) if grid is None else grid
    f = calibration_curve_fit(y, p, w)
    est = f(grid) if f is not None else np.full(len(grid), np.nan)
    curves = []
    for idx in boot_indices(len(y), B, seed, strata):
        fb = calibration_curve_fit(np.asarray(y)[idx], p[idx], None if w is None else np.asarray(w)[idx])
        if fb is not None:
            curves.append(fb(grid))
    curves = np.array(curves)
    return pd.DataFrame({"pred": grid, "obs": est, "lo": np.nanpercentile(curves, 2.5, axis=0),
                         "hi": np.nanpercentile(curves, 97.5, axis=0)})


def wilson(k, n, alpha=0.05):
    from statsmodels.stats.proportion import proportion_confint
    lo, hi = proportion_confint(k, n, alpha=alpha, method="wilson")
    return k / n, lo, hi


def bh_qvalues(p):
    p = np.asarray(p, float)
    n = np.sum(np.isfinite(p))
    q = np.full(len(p), np.nan)
    idx = np.where(np.isfinite(p))[0]
    order = idx[np.argsort(p[idx])]
    ranked = p[order] * n / (np.arange(1, n + 1))
    ranked = np.minimum.accumulate(ranked[::-1])[::-1]
    q[order] = np.minimum(ranked, 1)
    return q


In [ ]:
%%writefile {SRC_DIR}/rtw_models.py
"""Model definitions, tuning (development data only), OOF predictions and attributions (revision v2)."""
from __future__ import annotations

import time
import warnings
import zlib

import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from scipy.stats import loguniform, randint, uniform
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

from rtw_config import FAST, SEED

warnings.filterwarnings("ignore")

ONEHOT_MODELS = {"RidgeOH"}          # models that use the ordinal-one-hot design matrix


def design_seed(tag: str, task: str) -> int:
    """Design-specific seed so that random searches do not reuse one candidate list across designs."""
    return SEED + zlib.crc32(f"{tag}|{task}".encode()) % 10000


def make_estimator(name: str, params: dict | None = None, n_jobs: int = 1, seed: int = SEED) -> Pipeline:
    imp = SimpleImputer(strategy="median", add_indicator=True)
    if name in ("LR", "LR-small", "Status"):
        steps = [("imp", imp), ("sc", StandardScaler()), ("clf", LogisticRegression(penalty=None, max_iter=10000))]
    elif name == "Null":
        steps = [("imp", imp), ("clf", DummyClassifier(strategy="prior"))]
    elif name == "RidgeOH":
        steps = [("imp", imp), ("sc", StandardScaler()),
                 ("clf", LogisticRegression(penalty="l2", C=0.1, max_iter=10000))]
    elif name == "RF":
        steps = [("imp", imp), ("clf", RandomForestClassifier(n_estimators=500, n_jobs=n_jobs, random_state=seed))]
    elif name == "XGB":
        steps = [("imp", imp), ("clf", XGBClassifier(tree_method="hist", n_jobs=n_jobs, random_state=seed,
                                                     eval_metric="logloss", verbosity=0))]
    elif name == "LGBM":
        steps = [("imp", imp), ("clf", LGBMClassifier(n_jobs=n_jobs, random_state=seed, verbose=-1, subsample_freq=1))]
    elif name == "CatBoost":
        from catboost import CatBoostClassifier
        steps = [("imp", imp), ("clf", CatBoostClassifier(verbose=0, thread_count=n_jobs, random_seed=seed,
                                                          allow_writing_files=False))]
    elif name == "EBM":
        from interpret.glassbox import ExplainableBoostingClassifier
        steps = [("imp", imp), ("clf", ExplainableBoostingClassifier(n_jobs=n_jobs, random_state=seed, outer_bags=8))]
    else:
        raise ValueError(name)
    pipe = Pipeline(steps)
    if params:
        pipe.set_params(**params)
    return pipe


def search_space(name: str, reduced: bool = False):
    """(space, n_iter, kind). Widened ranges (v2): depth-1 trees, small/large leaves, wider learning rates."""
    if FAST:
        reduced = True
    if name == "RidgeOH":
        return ({"clf__C": list(np.logspace(-4, 1, 11))}, None, "grid")
    if name == "RF":
        return ({"clf__max_features": ["sqrt", 0.1, 0.2, 0.33, 0.5],
                 "clf__min_samples_leaf": [1, 5, 10, 20, 40, 80],
                 "clf__max_depth": [3, 4, 6, 8, 10, 14, None]}, 4 if FAST else (8 if reduced else 20), "random")
    if name == "XGB":
        return ({"clf__n_estimators": randint(50, 1501), "clf__learning_rate": loguniform(0.005, 0.3),
                 "clf__max_depth": randint(1, 7), "clf__min_child_weight": loguniform(1, 50),
                 "clf__subsample": uniform(0.5, 0.5), "clf__colsample_bytree": uniform(0.3, 0.7),
                 "clf__reg_lambda": loguniform(0.01, 50)}, 4 if FAST else (20 if reduced else 50), "random")
    if name == "LGBM":
        return ({"clf__n_estimators": randint(50, 1501), "clf__learning_rate": loguniform(0.005, 0.3),
                 "clf__num_leaves": randint(2, 65), "clf__min_child_samples": randint(5, 201),
                 "clf__subsample": uniform(0.5, 0.5), "clf__colsample_bytree": uniform(0.3, 0.7),
                 "clf__reg_lambda": uniform(0, 50)}, 4 if FAST else (20 if reduced else 50), "random")
    if name == "CatBoost":
        return ({"clf__iterations": randint(100, 1001), "clf__learning_rate": loguniform(0.01, 0.3),
                 "clf__depth": randint(1, 7), "clf__l2_leaf_reg": loguniform(1, 30)},
                3 if FAST else (8 if reduced else 20), "random")
    if name == "EBM":
        return ({"clf__interactions": [0, 10], "clf__learning_rate": [0.01, 0.04]}, None, "grid")
    return (None, None, None)


def make_folds(y, strata, k=5, seed=SEED):
    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=seed)
    return list(skf.split(np.zeros(len(y)), strata))


def tune(name, X, y, folds, n_jobs=4, reduced=False, seed=SEED):
    """Tune by CV log-loss. Returns (best_params, best_cv_logloss, seconds, n_candidates)."""
    t0 = time.time()
    space, n_iter, kind = search_space(name, reduced)
    if kind is None:
        est = make_estimator(name)
        s = cross_val_score(est, X, y, cv=folds, scoring="neg_log_loss", n_jobs=n_jobs)
        return {}, float(-s.mean()), time.time() - t0, 1
    inner_jobs = n_jobs if name in ("RF", "CatBoost", "EBM") else 1
    outer_jobs = 1 if name in ("RF", "CatBoost", "EBM") else n_jobs
    est = make_estimator(name, n_jobs=inner_jobs, seed=seed)
    if kind == "grid":
        gs = GridSearchCV(est, space, scoring="neg_log_loss", cv=folds, n_jobs=outer_jobs, refit=False)
    else:
        gs = RandomizedSearchCV(est, space, n_iter=n_iter, scoring="neg_log_loss", cv=folds,
                                n_jobs=outer_jobs, refit=False, random_state=seed)
    gs.fit(X, y)
    best = {k: (v.item() if hasattr(v, "item") else v) for k, v in gs.best_params_.items()}
    ncand = len(gs.cv_results_["params"])
    return best, float(-gs.best_score_), time.time() - t0, ncand


def oof_predict(name, params, X, y, folds, n_jobs=4, sample_weight=None):
    oof = np.full(len(y), np.nan)
    for tr, te in folds:
        est = make_estimator(name, params, n_jobs=n_jobs)
        kw = {} if sample_weight is None else {"clf__sample_weight": np.asarray(sample_weight)[tr]}
        est.fit(X.iloc[tr], np.asarray(y)[tr], **kw)
        oof[te] = est.predict_proba(X.iloc[te])[:, 1]
    return oof


def fit(name, params, X, y, n_jobs=4, sample_weight=None, seed=SEED):
    est = make_estimator(name, params, n_jobs=n_jobs, seed=seed)
    kw = {} if sample_weight is None else {"clf__sample_weight": np.asarray(sample_weight)}
    est.fit(X, np.asarray(y), **kw)
    return est


def ridge_cv(X, y, n_jobs=4, seed=SEED):
    """Ridge LR with C chosen by 5-fold CV log-loss on the given data (used for small-sample refits)."""
    folds = make_folds(y, y, seed=seed)
    best, ll, _, _ = tune("RidgeOH", X, y, folds, n_jobs=n_jobs)
    return fit("RidgeOH", best, X, y, n_jobs=n_jobs)


# ---- attributions (log-odds scale) -----------------------------------------------------------

def attributions(pipe: Pipeline, X: pd.DataFrame, max_rows: int | None = None, seed=SEED):
    """Exact additive attributions: TreeSHAP for tree ensembles, term contributions for EBM, coef*x for LR."""
    Xt_arr = pipe.named_steps["imp"].transform(X)
    names = list(pipe.named_steps["imp"].get_feature_names_out(X.columns))
    Xt = pd.DataFrame(Xt_arr, columns=names, index=X.index)
    if max_rows is not None and len(Xt) > max_rows:
        Xt = Xt.sample(max_rows, random_state=seed)
    clf = pipe.named_steps["clf"]
    cname = type(clf).__name__
    if cname == "LGBMClassifier":
        sv = clf.predict(Xt, pred_contrib=True)[:, :-1]
    elif cname == "XGBClassifier":
        import xgboost as xgb
        sv = clf.get_booster().predict(xgb.DMatrix(Xt.values), pred_contribs=True)[:, :-1]
    elif cname == "CatBoostClassifier":
        from catboost import Pool
        sv = clf.get_feature_importance(Pool(Xt.values), type="ShapValues")[:, :-1]
    elif cname == "ExplainableBoostingClassifier":
        terms = clf.eval_terms(Xt.values)          # per-term contributions (mains + interactions)
        sv = np.zeros((len(Xt), len(names)))
        for j, feats in enumerate(clf.term_features_):
            share = terms[:, j] / len(feats)
            for fidx in feats:
                sv[:, fidx] += share
    elif cname == "RandomForestClassifier":
        import shap
        v = shap.TreeExplainer(clf).shap_values(Xt.values, check_additivity=False)
        sv = v[1] if isinstance(v, list) else (v[:, :, 1] if np.ndim(v) == 3 else v)
    else:  # linear pipelines: coef * standardised x (centered)
        Z = pipe.named_steps["sc"].transform(Xt_arr) if "sc" in pipe.named_steps else Xt_arr
        Z = pd.DataFrame(Z, index=X.index).loc[Xt.index].values
        sv = Z * clf.coef_[0]
    return pd.DataFrame(sv, columns=names, index=Xt.index)


def aggregate_attr(sv: pd.DataFrame, groups: dict) -> pd.Series:
    g = {c: groups.get(c.replace("missingindicator_", ""), c.replace("missingindicator_", "")) for c in sv.columns}
    agg = sv.T.groupby(pd.Series(g)).sum().T
    return agg.abs().mean().sort_values(ascending=False)


In [ ]:
%%writefile {SRC_DIR}/rtw_develop.py
"""Development (tuning in the training cohorts) and prediction in the test cohort, revision v2."""
from __future__ import annotations

import json
import os
import time

import joblib
import numpy as np
import pandas as pd

from rtw_config import BASELINES, CANDIDATES, N_JOBS, SEED
from rtw_data import design_matrix, small_vars, status_columns, task_frame
from rtw_metrics import logloss
from rtw_models import ONEHOT_MODELS, design_seed, fit, make_folds, oof_predict, tune


def designs_for(d, task, variables=None):
    Xn, gn = design_matrix(d, task, variables, ordinal_onehot=False)
    Xo, go = design_matrix(d, task, variables, ordinal_onehot=True)
    return {"num": (Xn, gn), "oh": (Xo, go)}


def develop(a, task, train_cohorts=(1, 2), test_cohort=3, reduced=False, tag="primary", cache_dir=None,
            log=print, models=None, variables=None, select_repeats=2):
    """Tune each candidate (CV log-loss), compute OOF and test predictions, select the primary model by
    repeated-CV log-loss. Returns a dict (cached)."""
    models = list(models or CANDIDATES)
    cache = None if cache_dir is None else os.path.join(cache_dir, f"dev_{tag}_{task}.joblib")
    if cache and os.path.exists(cache):
        log(f"[develop] cache {os.path.basename(cache)}")
        return joblib.load(cache)
    t0 = time.time()
    d = task_frame(a, task)
    D = designs_for(d, task, variables)
    tr = d[d["cohort"].isin(train_cohorts)]
    te = d[d["cohort"] == test_cohort] if test_cohort is not None else d.iloc[:0]
    yd, yt = tr["y"].values, te["y"].values
    strata = tr["cohort"].values * 10 + yd
    folds = make_folds(yd, strata)
    folds_rep = [make_folds(yd, strata, seed=SEED + 101 * r) for r in range(1, select_repeats)]
    seed = design_seed(tag, task)
    res = {"task": task, "tag": tag, "train_cohorts": list(train_cohorts), "test_cohort": test_cohort,
           "dev_index": tr.index.values, "test_index": te.index.values, "y_dev": yd, "y_test": yt,
           "dev_cohort": tr["cohort"].values, "params": {}, "cv_logloss": {}, "rep_logloss": {},
           "tune_seconds": {}, "n_candidates": {}, "oof": {}, "test": {}, "estimators": {},
           "groups": {k: v[1] for k, v in D.items()}, "columns": {k: list(v[0].columns) for k, v in D.items()}}
    for m in models:
        X = D["oh" if m in ONEHOT_MODELS else "num"][0]
        Xd, Xt = X.loc[tr.index], X.loc[te.index]
        params, ll, sec, nc = tune(m, Xd, yd, folds, n_jobs=N_JOBS, reduced=reduced, seed=seed)
        res["params"][m], res["cv_logloss"][m], res["tune_seconds"][m], res["n_candidates"][m] = params, ll, sec, nc
        res["oof"][m] = oof_predict(m, params, Xd, yd, folds, n_jobs=N_JOBS)
        rep = [logloss(yd, res["oof"][m])]
        for fr in folds_rep:
            rep.append(logloss(yd, oof_predict(m, params, Xd, yd, fr, n_jobs=N_JOBS)))
        res["rep_logloss"][m] = float(np.mean(rep))
        est = fit(m, params, Xd, yd, n_jobs=N_JOBS, seed=seed)
        res["estimators"][m] = est
        if len(te):
            res["test"][m] = est.predict_proba(Xt)[:, 1]
        log(f"[develop {tag} {task}] {m}: CV ll {ll:.4f}; repeated-CV ll {res['rep_logloss'][m]:.4f} "
            f"({sec:.0f}s, {nc} cand.) {params}")
    # baselines -------------------------------------------------------------------------------
    Xn = D["num"][0]
    for b in BASELINES[task]:
        if b == "Null":
            cols = Xn.columns[:1].tolist()
        elif b == "Status":
            cols = status_columns(Xn)
        else:
            cols = [c for c in Xn.columns if D["num"][1][c] in small_vars(task)]
        name = "Null" if b == "Null" else ("Status" if b == "Status" else "LR-small")
        res["oof"][b] = oof_predict(name, None, Xn.loc[tr.index, cols], yd, folds)
        est = fit(name, None, Xn.loc[tr.index, cols], yd)
        res["estimators"][b] = est
        res[f"cols_{b}"] = cols
        if len(te):
            res["test"][b] = est.predict_proba(Xn.loc[te.index, cols])[:, 1]
    if task == "P" and len(te):
        res["status_flag_test"] = (te["employed"].values == 0).astype(int)   # status rule: flag not employed at w1
    res["primary"] = min(res["rep_logloss"], key=res["rep_logloss"].get)
    trees = {m: v for m, v in res["rep_logloss"].items() if m in ("RF", "XGB", "LGBM", "CatBoost", "EBM")}
    res["attr_model"] = min(trees, key=trees.get) if trees else None
    res["seconds"] = time.time() - t0
    log(f"[develop {tag} {task}] primary={res['primary']} (repeated-CV log-loss) ({res['seconds']:.0f}s)")
    if cache:
        joblib.dump(res, cache)
    return res


def params_table(res, repeats=2):
    """Tuning summary. Round 2 adds the two components of the repeated-CV log-loss: repeat 1 reuses the tuning folds
    (OOF log-loss), repeat 2 uses independent folds (= 2 x mean - repeat 1 when select_repeats = 2), and flags the
    model each rule would select (single tuning-CV rule of v1; repeated-CV rule of v2.0; independent repeat only)."""
    rows = []
    for m in res["params"]:
        rep1 = logloss(res["y_dev"], res["oof"][m])
        rows.append({"task": res["task"], "design": res["tag"], "model": m, "cv_logloss": res["cv_logloss"][m],
                     "repeated_cv_logloss": res["rep_logloss"][m], "logloss_repeat1_tuning_folds": rep1,
                     "logloss_repeat2_independent_folds": (2 * res["rep_logloss"][m] - rep1) if repeats == 2 else np.nan,
                     "n_candidates": res["n_candidates"][m], "tune_seconds": round(res["tune_seconds"][m], 1),
                     "params": json.dumps(res["params"][m], default=str)})
    df = pd.DataFrame(rows)
    for col, flag in (("cv_logloss", "selected_v1_single_cv"), ("repeated_cv_logloss", "selected_v2_repeated_cv"),
                      ("logloss_repeat2_independent_folds", "selected_independent_repeat")):
        df[flag] = (df[col] == df[col].min()) if df[col].notna().any() else False
    return df


In [ ]:
%%writefile {SRC_DIR}/rtw_shift.py
"""Dataset-shift diagnostics (revision v2).

* outcome-prevalence change and transition probabilities (unweighted and survey-weighted)
* conditional cohort effects (pooled logistic model with cohort indicators)
* test of whether the outcome improves a domain classifier given x (conditional/concept shift)
* adversarial validation per cohort pair (full and comparability-restricted predictor sets)
* DISDE-type decomposition (Cai, Namkoong & Yadlowsky): performance change from source to target split into
  X-shift on the source side, Y|X shift on the shared support and X-shift on the target side; the shared
  distribution has density proportional to p_S(x) p_T(x) / (p_S(x) + p_T(x)), so the weights are bounded in (0, 1)
  and need no truncation. The domain classifier is re-estimated inside every bootstrap resample.
* model-based (case-mix) expected AUROC; full importance-weighting truncation sensitivity, ESS and balance
"""
from __future__ import annotations

import numpy as np
import pandas as pd
from scipy.stats import chi2
from sklearn.model_selection import StratifiedKFold

from rtw_config import B_SHIFT, N_JOBS, PREDICTORS, RESTRICTED_DROP_B, SEED, VAR_LABELS
from rtw_data import design_matrix, task_frame
from rtw_metrics import (EPS, auroc, brier, cal_slope, citl, logloss, newton_logistic, wilson)
from rtw_models import aggregate_attr, make_estimator


# ---- prevalence ------------------------------------------------------------------------------

def prevalence_table(a):
    rows = []
    for task in ("P", "A1", "A2"):
        d = task_frame(a, task)
        for c in (1, 2, 3):
            s = d[d["cohort"] == c]
            p, lo, hi = wilson(int(s["y"].sum()), len(s))
            pw = np.average(s["y"], weights=s["wt_w2"])
            rows.append({"task": task, "cohort": c, "n": len(s), "employed_w2": int(s["y"].sum()), "prev": p,
                         "lo": lo, "hi": hi, "prev_weighted": pw})
    return pd.DataFrame(rows)


# ---- conditional cohort effects --------------------------------------------------------------

def cohort_effects(a, task):
    import statsmodels.api as sm
    d = task_frame(a, task)
    X, _ = design_matrix(d, task)
    X = X.fillna(X.median())
    X = X.loc[:, X.std() > 0]
    X = (X - X.mean()) / X.std()
    X["C2"] = (d["cohort"] == 2).astype(float)
    X["C3"] = (d["cohort"] == 3).astype(float)
    mod = sm.Logit(d["y"].values, sm.add_constant(X)).fit(disp=0, maxiter=300)
    ci = mod.conf_int()
    cov = mod.cov_params()
    rows = []
    for k in ("C2", "C3"):
        rows.append({"task": task, "contrast": f"{k} vs C1", "log_odds": mod.params[k], "lo": ci.loc[k, 0],
                     "hi": ci.loc[k, 1], "p": mod.pvalues[k]})
    diff = mod.params["C3"] - mod.params["C2"]
    se = np.sqrt(cov.loc["C3", "C3"] + cov.loc["C2", "C2"] - 2 * cov.loc["C2", "C3"])
    rows.append({"task": task, "contrast": "C3 vs C2", "log_odds": diff, "lo": diff - 1.96 * se,
                 "hi": diff + 1.96 * se, "p": 2 * (1 - __import__("scipy").stats.norm.cdf(abs(diff / se)))})
    # unadjusted (crude) log-odds differences for comparison
    for (c1, c0) in ((2, 1), (3, 1), (3, 2)):
        y1, y0 = d.loc[d.cohort == c1, "y"].mean(), d.loc[d.cohort == c0, "y"].mean()
        rows.append({"task": task, "contrast": f"crude C{c1} vs C{c0}",
                     "log_odds": np.log(y1 / (1 - y1)) - np.log(y0 / (1 - y0)), "lo": np.nan, "hi": np.nan, "p": np.nan})
    return pd.DataFrame(rows)


def outcome_in_domain_test(a, task, src, tgt=(3,)):
    """LRT: does y improve a logistic domain classifier (target vs source) given x? (conditional shift test)."""
    d = task_frame(a, task)
    d = d[d["cohort"].isin(list(src) + list(tgt))]
    X, _ = design_matrix(d, task)
    X = X.fillna(X.median())
    X = X.loc[:, X.std() > 0]
    Z = ((X - X.mean()) / X.std()).values
    z = d["cohort"].isin(tgt).astype(int).values
    X0 = np.column_stack([np.ones(len(z)), Z])
    X1 = np.column_stack([X0, d["y"].values])
    def ll(Xm):
        b, ok = newton_logistic(Xm, z)
        mu = np.clip(1 / (1 + np.exp(-(Xm @ b))), EPS, 1 - EPS)
        return np.sum(z * np.log(mu) + (1 - z) * np.log(1 - mu)), b
    l0, _ = ll(X0)
    l1, b1 = ll(X1)
    stat = 2 * (l1 - l0)
    return {"task": task, "source": "+".join(f"C{c}" for c in src), "target": "C3", "coef_y": b1[-1],
            "lrt": stat, "p": float(chi2.sf(max(stat, 0), 1))}


# ---- domain classifiers ----------------------------------------------------------------------

def _lgbm_domain():
    from lightgbm import LGBMClassifier
    return LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=15, min_child_samples=40,
                          random_state=SEED, n_jobs=N_JOBS, verbose=-1)


def domain_probs(Xs, Xt, seed=SEED, kind="LR"):
    """Cross-fitted P(target | x) for the stacked source and target rows (5 folds)."""
    Xa = pd.concat([Xs, Xt])
    z = np.r_[np.zeros(len(Xs)), np.ones(len(Xt))].astype(int)
    pz = np.zeros(len(z))
    for tr, te in StratifiedKFold(5, shuffle=True, random_state=seed).split(Xa, z):
        est = make_estimator("LR") if kind == "LR" else _lgbm_domain()
        est.fit(Xa.iloc[tr], z[tr])
        pz[te] = est.predict_proba(Xa.iloc[te])[:, 1]
    pz = np.clip(pz, EPS, 1 - EPS)
    return pz[: len(Xs)], pz[len(Xs):], auroc(z, pz)


def density_ratio(pi, ns, nt):
    """r(x) = p_T(x) / p_S(x) from P(target | x) with the sample-size prior corrected."""
    return pi / (1 - pi) * (ns / nt)


def adversarial(a, pairs, B=1000, restricted=False):
    rows, drivers = [], []
    variables = [v for v in PREDICTORS if v not in RESTRICTED_DROP_B] if restricted else None
    d = task_frame(a, "P")
    X, groups = design_matrix(d, "P", variables)
    for src, tgt in pairs:
        m = d["cohort"].isin(list(src) + list(tgt))
        Xm = X[m]
        z = d.loc[m, "cohort"].isin(tgt).astype(int).values
        oof = np.zeros(len(z))
        for tr, te in StratifiedKFold(5, shuffle=True, random_state=SEED).split(Xm, z):
            oof[te] = _lgbm_domain().fit(Xm.iloc[tr], z[tr]).predict_proba(Xm.iloc[te])[:, 1]
        rng = np.random.default_rng(SEED)
        bs = [auroc(z[i], oof[i]) for i in (rng.integers(0, len(z), len(z)) for _ in range(B))]
        lab = f"{'+'.join(f'C{c}' for c in src)} vs {'+'.join(f'C{c}' for c in tgt)}"
        rows.append({"contrast": lab, "predictors": "restricted" if restricted else "all", "n": len(z),
                     "AUROC": auroc(z, oof), "lo": np.percentile(bs, 2.5), "hi": np.percentile(bs, 97.5)})
        clf = _lgbm_domain().fit(Xm, z)
        sv = pd.DataFrame(clf.predict(Xm, pred_contrib=True)[:, :-1], columns=Xm.columns)
        imp = aggregate_attr(sv, groups)
        drivers.append(pd.DataFrame({"contrast": lab, "predictors": rows[-1]["predictors"], "variable": imp.index,
                                     "label": [VAR_LABELS.get(v, v) for v in imp.index],
                                     "mean_abs_shap": imp.values}))
    return pd.DataFrame(rows), pd.concat(drivers)


# ---- DISDE-type decomposition ----------------------------------------------------------------

def _metric(name, y, p, w=None):
    p = np.clip(p, EPS, 1 - EPS)
    if name == "Prevalence":
        return float(np.average(y, weights=w))
    if name == "Brier":
        return brier(y, p, w)
    if name == "LogLoss":
        return logloss(y, p, w)
    if name == "AUROC":
        return auroc(y, p, w)
    if name == "CITL":
        return citl(y, p, w)
    if name == "Slope":
        return cal_slope(y, p, w)
    raise ValueError(name)


DISDE_METRICS = ("Prevalence", "Brier", "LogLoss", "AUROC", "CITL", "Slope")


def _disde_once(ys, ps_d, yt, pt_d, pis, pit, ns, nt):
    rs, rt = density_ratio(pis, ns, nt), density_ratio(pit, ns, nt)
    ws = rs / (1 + rs)          # shared / source density ratio (up to a constant)
    wt = 1 / (1 + rt)           # shared / target density ratio (up to a constant)
    out = {}
    for model in ps_d:
        ps, pt = ps_d[model], pt_d[model]
        for m in DISDE_METRICS:
            if m == "Prevalence" and model != list(ps_d)[0]:
                continue
            mP, mSP = _metric(m, ys, ps), _metric(m, ys, ps, ws)
            mSQ, mQ = _metric(m, yt, pt, wt), _metric(m, yt, pt)
            out[(model, m)] = (mP, mSP, mSQ, mQ)
    ess_s = ws.sum() ** 2 / np.sum(ws ** 2)
    ess_t = wt.sum() ** 2 / np.sum(wt ** 2)
    return out, ess_s, ess_t


def disde(Xs, ys, ps_d, Xt, yt, pt_d, B=B_SHIFT, seed=SEED, kind="LR"):
    """Decompose metric(target) - metric(source) into X-shift (source side), Y|X shift (shared support) and
    X-shift (target side) for every model in ps_d/pt_d (dicts of source OOF / target predictions).
    The domain classifier is re-estimated inside each bootstrap resample (source and target resampled)."""
    ys, yt = np.asarray(ys), np.asarray(yt)
    ps_d = {k: np.asarray(v) for k, v in ps_d.items()}
    pt_d = {k: np.asarray(v) for k, v in pt_d.items()}
    pis, pit, dauc = domain_probs(Xs, Xt, seed, kind)
    point, ess_s, ess_t = _disde_once(ys, ps_d, yt, pt_d, pis, pit, len(ys), len(yt))
    rng = np.random.default_rng(seed)
    boots = {k: [] for k in point}
    for b in range(B):
        i_s = rng.integers(0, len(ys), len(ys))
        i_t = rng.integers(0, len(yt), len(yt))
        bps, bpt, _ = domain_probs(Xs.iloc[i_s], Xt.iloc[i_t], seed + b + 1, kind)
        o, _, _ = _disde_once(ys[i_s], {k: v[i_s] for k, v in ps_d.items()}, yt[i_t],
                              {k: v[i_t] for k, v in pt_d.items()}, bps, bpt, len(i_s), len(i_t))
        for k in point:
            boots[k].append(o[k])
    rows = []
    for (model, m), (mP, mSP, mSQ, mQ) in point.items():
        comps = {"total": mQ - mP, "X_shift_source_side": mSP - mP, "YgivenX_shift": mSQ - mSP,
                 "X_shift_target_side": mQ - mSQ}
        bb = np.array(boots[(model, m)], float)
        bcomp = {"total": bb[:, 3] - bb[:, 0], "X_shift_source_side": bb[:, 1] - bb[:, 0],
                 "YgivenX_shift": bb[:, 2] - bb[:, 1], "X_shift_target_side": bb[:, 3] - bb[:, 2]}
        for k, v in comps.items():
            s = bcomp[k][np.isfinite(bcomp[k])]
            # the prevalence decomposition uses observed outcomes only (no model): label it model-free (round 2)
            rows.append({"model": "model-free" if m == "Prevalence" else model, "metric": m, "component": k,
                         "estimate": v,
                         "lo": np.percentile(s, 2.5) if len(s) else np.nan,
                         "hi": np.percentile(s, 97.5) if len(s) else np.nan,
                         "source_value": mP, "shared_source_value": mSP, "shared_target_value": mSQ,
                         "target_value": mQ})
    info = {"domain_auroc": dauc, "ess_source_shared": ess_s, "ess_target_shared": ess_t,
            "n_source": len(ys), "n_target": len(yt), "B": B, "classifier": kind}
    return pd.DataFrame(rows), info


# ---- case-mix benchmark ----------------------------------------------------------------------

def casemix_auroc(p, B=200, seed=SEED):
    """Model-based expected AUROC if the model were calibrated in this population (Vergouwe et al.)."""
    rng = np.random.default_rng(seed)
    p = np.asarray(p, float)
    return float(np.mean([auroc(rng.binomial(1, p), p) for _ in range(B)]))


# ---- full importance weighting: truncation sensitivity and balance ---------------------------

def iw_sensitivity(Xs, ys, ps, Xt, yt, pt, caps=(95, 99, 99.9, None)):
    rows = []
    for kind in ("LR", "LGBM"):
        pis, pit, dauc = domain_probs(Xs, Xt, SEED, kind)
        r = density_ratio(pis, len(ys), len(yt))
        Zs = Xs.fillna(Xs.median())
        Zt = Xt.fillna(Xs.median())
        sd = np.sqrt((Zs.var() + Zt.var()) / 2).replace(0, np.nan)
        smd_before = ((Zt.mean() - Zs.mean()) / sd).abs()
        for cap in caps:
            w = np.minimum(r, np.percentile(r, cap)) if cap is not None else r.copy()
            w = w / w.mean()
            mw = (Zs.mul(w, axis=0)).sum() / w.sum()
            smd_after = ((Zt.mean() - mw) / sd).abs()
            rows.append({"classifier": kind, "domain_auroc": dauc, "cap_pct": "none" if cap is None else cap,
                         "ESS": w.sum() ** 2 / np.sum(w ** 2), "max_w": float(w.max()),
                         "prev_dev": float(np.mean(ys)), "prev_weighted": float(np.average(ys, weights=w)),
                         "prev_target": float(np.mean(yt)),
                         "max_abs_smd_before": float(smd_before.max()), "n_smd_gt_0.1_before": int((smd_before > 0.1).sum()),
                         "max_abs_smd_after": float(smd_after.max()), "n_smd_gt_0.1_after": int((smd_after > 0.1).sum()),
                         "weighted_CITL": citl(ys, np.clip(ps, EPS, 1 - EPS), w),
                         "weighted_Slope": cal_slope(ys, np.clip(ps, EPS, 1 - EPS), w),
                         "weighted_AUROC": auroc(ys, ps, w),
                         "target_CITL": citl(yt, np.clip(pt, EPS, 1 - EPS)), "target_Slope": cal_slope(yt, pt),
                         "target_AUROC": auroc(yt, pt)})
    return pd.DataFrame(rows)


In [ ]:
%%writefile {SRC_DIR}/rtw_update.py
"""Model updating (revision v2).

(i) Label-free importance-weighted recalibration (IW-recal): a logistic recalibration a + b*logit(p) fitted on the
    source out-of-fold predictions, weighted by density ratios to the UNLABELLED target covariates; no target outcomes.
(ii) Lagged labelled updating: model developed in C1, updated with random C2 samples, evaluated in C3.
(iii) Within-C3 split design (v1 design, corrected): update half / evaluation half of C3.
Closed testing (Vergouwe et al. 2017) chooses among no update, intercept and logistic recalibration by LR tests.
"""
from __future__ import annotations

import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from scipy.stats import chi2

from rtw_config import B_SHIFT, N_JOBS, SEED
from rtw_metrics import (EPS, auroc, brier, cal_slope, citl, expit, ici, logistic_recal, logit, newton_logistic)
from rtw_models import fit, ridge_cv
from rtw_shift import density_ratio, domain_probs

UPD_METRICS = ("AUROC", "Brier", "CITL", "Slope", "ICI")


def _m(y, p):
    p = np.clip(np.asarray(p, float), EPS, 1 - EPS)
    i, _, _ = ici(y, p)
    return {"AUROC": auroc(y, p), "Brier": brier(y, p), "CITL": citl(y, p), "Slope": cal_slope(y, p), "ICI": i}


def recal_params(y, p, w=None, kind="logistic"):
    lp = logit(p)
    if kind == "intercept":
        b, ok = logistic_recal(y, lp, w, slope=False)
        return (float(b[0]), 1.0) if ok else (0.0, 1.0)
    b, ok = logistic_recal(y, lp, w, slope=True)
    return (float(b[0]), float(b[1])) if ok else (0.0, 1.0)


def apply_recal(p, ab):
    return expit(ab[0] + ab[1] * logit(p))


def closed_test(y, p, alpha=0.05):
    """Vergouwe et al. (2017) closed testing among no update (M0), intercept (M1), logistic recalibration (M2)."""
    y = np.asarray(y, float)
    lp = logit(p)
    def ll(mu):
        mu = np.clip(mu, EPS, 1 - EPS)
        return np.sum(y * np.log(mu) + (1 - y) * np.log(1 - mu))
    l0 = ll(expit(lp))
    b1, _ = logistic_recal(y, lp, None, slope=False)
    l1 = ll(expit(b1[0] + lp))
    b2, _ = logistic_recal(y, lp, None, slope=True)
    l2 = ll(expit(b2[0] + b2[1] * lp))
    if chi2.sf(2 * (l2 - l0), 2) >= alpha:
        return "none", (0.0, 1.0)
    if chi2.sf(2 * (l2 - l1), 1) < alpha:
        return "logistic", (float(b2[0]), float(b2[1]))
    return "intercept", (float(b1[0]), 1.0)


# ---- (i) label-free importance-weighted recalibration -----------------------------------------

def iw_recal_eval(Xs, ys, ps_d, Xt, yt, pt_d, B=B_SHIFT, seed=SEED, cap=99):
    """For every model: no update, unweighted OOF recalibration, IW intercept and IW logistic recalibration
    (no target labels used). Bootstrap resamples source and target; domain weights and recalibrations are
    re-estimated in every resample."""
    ys, yt = np.asarray(ys), np.asarray(yt)
    ps_d = {k: np.asarray(v) for k, v in ps_d.items()}
    pt_d = {k: np.asarray(v) for k, v in pt_d.items()}

    def once(i_s, i_t, sd):
        pis, pit, _ = domain_probs(Xs.iloc[i_s], Xt.iloc[i_t], sd)
        r = density_ratio(pis, len(i_s), len(i_t))
        w = np.minimum(r, np.percentile(r, cap))
        w = w / w.mean()
        y_s, y_t = ys[i_s], yt[i_t]
        res, pars = {}, {}
        for model in ps_d:
            p_s, p_t = ps_d[model][i_s], pt_d[model][i_t]
            ab = recal_params(y_s, p_s, w)
            preds = {"No update": p_t,
                     "OOF recalibration (unweighted)": apply_recal(p_t, recal_params(y_s, p_s)),
                     "IW intercept recalibration": apply_recal(p_t, recal_params(y_s, p_s, w, "intercept")),
                     "IW logistic recalibration": apply_recal(p_t, ab)}
            res[model] = {k: _m(y_t, v) for k, v in preds.items()}
            pars[model] = ab
        return res, pars
    point, pars = once(np.arange(len(ys)), np.arange(len(yt)), seed)
    rng = np.random.default_rng(seed + 7)
    boots = [once(rng.integers(0, len(ys), len(ys)), rng.integers(0, len(yt), len(yt)), seed + b + 11)[0]
             for b in range(B)]
    rows = []
    for model in point:
        for meth in point[model]:
            for k in UPD_METRICS:
                s = np.array([bb[model][meth][k] for bb in boots], float)
                s0 = np.array([bb[model]["No update"][k] for bb in boots], float)
                d = s - s0
                if k in ("CITL",):
                    d = np.abs(s) - np.abs(s0)
                if k == "Slope":
                    d = np.abs(s - 1) - np.abs(s0 - 1)
                okd = np.isfinite(d)
                est_d = point[model][meth][k] - point[model]["No update"][k]
                if k == "CITL":
                    est_d = abs(point[model][meth][k]) - abs(point[model]["No update"][k])
                if k == "Slope":
                    est_d = abs(point[model][meth][k] - 1) - abs(point[model]["No update"][k] - 1)
                rows.append({"model": model, "method": meth, "metric": k, "estimate": point[model][meth][k],
                             "lo": np.nanpercentile(s, 2.5), "hi": np.nanpercentile(s, 97.5),
                             "improvement_vs_none": est_d,
                             "imp_lo": np.percentile(d[okd], 2.5) if okd.sum() else np.nan,
                             "imp_hi": np.percentile(d[okd], 97.5) if okd.sum() else np.nan,
                             "iw_a": pars[model][0], "iw_b": pars[model][1]})
    return pd.DataFrame(rows)


# ---- (ii) lagged labelled updating ------------------------------------------------------------

def _lag_rep(r, model, params, X1, y1, X2, y2, p2, X3, y3, p3, Xoh2, Xoh3, sizes):
    rng = np.random.default_rng(SEED + 3000 + r)
    rows = []

    def rec(n, meth, p, extra=None):
        d = {"rep": r, "n": n, "method": meth, **_m(y3, p)}
        if extra:
            d.update(extra)
        rows.append(d)
    for n in sizes:
        if n == "all":
            if r > 0:
                continue
            idx = np.arange(len(y2))
        else:
            idx = rng.choice(len(y2), n, replace=False)
        nn = len(idx)
        rec(nn, "Intercept recalibration", apply_recal(p3, recal_params(y2[idx], p2[idx], kind="intercept")))
        rec(nn, "Logistic recalibration", apply_recal(p3, recal_params(y2[idx], p2[idx])))
        choice, ab = closed_test(y2[idx], p2[idx])
        rec(nn, "Closed testing", apply_recal(p3, ab), {"choice": choice})
        est = fit(model, params, pd.concat([X1, X2.iloc[idx]]), np.r_[y1, y2[idx]], n_jobs=1)
        rec(nn, "Pooled refit (C1 + C2 sample)", est.predict_proba(X3)[:, 1])
        try:
            rr = ridge_cv(Xoh2.iloc[idx], y2[idx], n_jobs=1, seed=SEED + r)
            rec(nn, "Ridge refit on C2 sample only", rr.predict_proba(Xoh3)[:, 1])
        except Exception:  # noqa: BLE001 - e.g. a single class in a tiny sample
            pass
    return rows


def lagged_updating(model, params, X1, y1, X2, y2, X3, y3, Xoh2, Xoh3, base_est, R=50,
                    sizes=(100, 250, 500, 1000, "all")):
    p2 = base_est.predict_proba(X2)[:, 1]
    p3 = base_est.predict_proba(X3)[:, 1]
    sizes = [s for s in sizes if s == "all" or s <= len(y2)]
    reps = Parallel(n_jobs=N_JOBS)(delayed(_lag_rep)(r, model, params, X1, y1, X2, y2, p2, X3, y3, p3, Xoh2, Xoh3,
                                                     sizes) for r in range(R))
    long = pd.DataFrame([row for rep in reps for row in rep])
    base = _m(y3, p3)
    long = pd.concat([pd.DataFrame([{"rep": 0, "n": 0, "method": "No update (C1 model)", **base}]), long],
                     ignore_index=True)
    return long


# ---- (iii) within-C3 split design --------------------------------------------------------------

def _c3_rep(r, model, params, Xd, yd, Xt, yt, p_base, Xoht, sizes):
    rng = np.random.default_rng(SEED + 1000 + r)
    pos, neg = np.where(yt == 1)[0], np.where(yt == 0)[0]
    rng.shuffle(pos)
    rng.shuffle(neg)
    pool = np.r_[pos[: len(pos) // 2], neg[: len(neg) // 2]]
    hold = np.r_[pos[len(pos) // 2:], neg[len(neg) // 2:]]
    yh = yt[hold]
    rows = [{"rep": r, "n": 0, "method": "No update", **_m(yh, p_base[hold])}]
    for n in sizes:
        if n > len(pool):
            continue
        sub = rng.choice(pool, n, replace=False)
        for kind, lab in (("intercept", "Intercept recalibration"), ("logistic", "Logistic recalibration")):
            rows.append({"rep": r, "n": n, "method": lab,
                         **_m(yh, apply_recal(p_base[hold], recal_params(yt[sub], p_base[sub], kind=kind)))})
        ch, ab = closed_test(yt[sub], p_base[sub])
        rows.append({"rep": r, "n": n, "method": "Closed testing", "choice": ch, **_m(yh, apply_recal(p_base[hold], ab))})
        est = fit(model, params, Xt.iloc[sub], yt[sub], n_jobs=1)
        rows.append({"rep": r, "n": n, "method": "Refit on C3 sample (development hyperparameters)",
                     **_m(yh, est.predict_proba(Xt.iloc[hold])[:, 1])})
        try:
            rr = ridge_cv(Xoht.iloc[sub], yt[sub], n_jobs=1, seed=SEED + r)
            rows.append({"rep": r, "n": n, "method": "Ridge refit on C3 sample (CV-tuned)",
                         **_m(yh, rr.predict_proba(Xoht.iloc[hold])[:, 1])})
        except Exception:  # noqa: BLE001
            pass
        est = fit(model, params, pd.concat([Xd, Xt.iloc[sub]]), np.r_[yd, yt[sub]], n_jobs=1)
        rows.append({"rep": r, "n": n, "method": "Pooled refit (dev + C3 sample)",
                     **_m(yh, est.predict_proba(Xt.iloc[hold])[:, 1])})
    return rows


def within_c3_updating(model, params, Xd, yd, Xt, yt, p_base, Xoht, R=50, sizes=(100, 250, 500, 1000)):
    reps = Parallel(n_jobs=N_JOBS)(delayed(_c3_rep)(r, model, params, Xd, yd, Xt, yt, p_base, Xoht, sizes)
                                   for r in range(R))
    return pd.DataFrame([row for rep in reps for row in rep])


def summarise_updating(long, baseline_label, tol=1e-10):
    """Median and 2.5-97.5 percentiles; paired comparisons with the baseline of the same repetition.

    An update counts as better only if it improves the metric by more than `tol` (2026-10-15 correction): when
    closed testing keeps the model, apply_recal(p, (0, 1)) returns expit(logit(p)), which differs from p only by
    floating-point noise, and intercept recalibration cannot change the calibration slope; strict '<' comparisons
    had counted such ties as improvements."""
    base = long[long["method"] == baseline_label].set_index("rep")
    rows = []
    for (meth, n), d in long.groupby(["method", "n"], sort=False):
        d = d.set_index("rep")
        if len(base) == 1:
            b = pd.DataFrame([base.iloc[0]] * len(d), index=d.index)
        else:
            b = base.reindex(d.index)
        row = {"method": meth, "n": int(n), "reps": len(d)}
        for k in UPD_METRICS:
            v = d[k].astype(float)
            row[f"{k}_median"] = v.median()
            row[f"{k}_p2_5"] = v.quantile(0.025)
            row[f"{k}_p97_5"] = v.quantile(0.975)
            row[f"{k}_n_undefined"] = int(v.isna().sum())
            row[f"{k}_median_diff_vs_base"] = (v - b[k].astype(float)).median()
        row["share_better_absCITL"] = float(np.mean(b["CITL"].abs() - d["CITL"].abs() > tol))
        row["share_better_slope"] = float(np.mean((b["Slope"] - 1).abs() - (d["Slope"] - 1).abs() > tol))
        row["share_better_Brier"] = float(np.mean(b["Brier"] - d["Brier"] > tol))
        row["share_better_ICI"] = float(np.mean(b["ICI"] - d["ICI"] > tol))
        row["mean_absCITL"] = float(d["CITL"].abs().mean())
        if "choice" in d:
            ch = d["choice"].dropna()
            for c in ("none", "intercept", "logistic"):
                row[f"closed_choice_{c}"] = float(np.mean(ch == c)) if len(ch) else np.nan
        rows.append(row)
    return pd.DataFrame(rows)


In [ ]:
%%writefile {SRC_DIR}/rtw_fair.py
"""Exploratory fairness audit (revision v2).

Flag = predicted risk of the undesired event (1 - p) >= threshold (development prevalence of the event).
Per group: AUROC, CITL, O/E, TPR (flagged | event), FPR (flagged | no event), FNR, PPV, NPV, and the TPR implied by
the model's own probabilities if it were calibrated in the group (sufficiency check, cf. Kleinberg et al.).
Gaps vs a reference group with bootstrap CIs and two-sided bootstrap p-values -> Benjamini-Hochberg q-values.
"""
from __future__ import annotations

import numpy as np
import pandas as pd

from rtw_metrics import EPS, auroc, bh_qvalues, boot_indices, citl

FAIR_GROUPS = {"Sex": ("sex_grp", "Male"), "Age group": ("age_lab", "<=39"),
               "Disability": ("disab_grp", "Mild (8-14)"), "Accident type": ("acc_grp", "Injury"),
               "Job status": ("job_grp", "Regular")}
GAP_METRICS = ("AUROC", "CITL", "TPR", "FPR", "PPV", "NPV")


def _gm(y, p, w, pt):
    event = (y == 0)
    r = 1 - p
    flag = r >= pt
    wv = np.ones(len(y)) if w is None else w
    def wm(mask, val):
        return np.sum(wv[mask] * val[mask]) / np.sum(wv[mask]) if np.sum(wv[mask]) > 0 else np.nan
    out = {"n": int(len(y)), "observed": float(np.average(y, weights=wv)), "mean_pred": float(np.average(p, weights=wv))}
    out["OE"] = out["observed"] / out["mean_pred"]
    out["AUROC"] = auroc(y, p, w) if 0 < y.mean() < 1 else np.nan
    out["CITL"] = citl(y, np.clip(p, EPS, 1 - EPS), w) if 0 < y.mean() < 1 else np.nan
    out["TPR"] = wm(event, flag.astype(float))
    out["FPR"] = wm(~event, flag.astype(float))
    out["FNR"] = 1 - out["TPR"] if np.isfinite(out["TPR"]) else np.nan
    out["PPV"] = wm(flag, event.astype(float))
    out["NPV"] = wm(~flag, (~event).astype(float))
    out["TPR_implied"] = float(np.sum(wv * r * flag) / np.sum(wv * r))
    out["flag_rate"] = float(np.average(flag, weights=wv))
    return out


def fairness(te, y, p, task, model, pt, B=1000, seed=2026, w=None, strata=None, label="unweighted"):
    y = np.asarray(y, int)
    p = np.asarray(p, float)
    idxs = boot_indices(len(y), B, seed, strata)
    rows = []
    for gname, (col, ref) in FAIR_GROUPS.items():
        g = te[col].values
        levels = [lv for lv in pd.unique(g)]
        def all_levels(idx):
            res = {}
            for lv in levels:
                m = idx[g[idx] == lv]
                if len(m) >= 5:
                    res[lv] = _gm(y[m], p[m], None if w is None else np.asarray(w)[m], pt)
            return res
        point = all_levels(np.arange(len(y)))
        bs = [all_levels(i) for i in idxs]
        for lv, mv in point.items():
            r = {"task": task, "model": model, "weighting": label, "threshold": pt, "group": gname, "level": lv,
                 "reference": ref, **mv}
            for k in GAP_METRICS + ("TPR_implied",):
                s = np.array([b[lv][k] for b in bs if lv in b], float)
                r[f"{k}_lo"], r[f"{k}_hi"] = np.nanpercentile(s, 2.5), np.nanpercentile(s, 97.5)
            if lv != ref and ref in point:
                for k in GAP_METRICS:
                    gap = np.array([b[lv][k] - b[ref][k] for b in bs if lv in b and ref in b], float)
                    gap = gap[np.isfinite(gap)]
                    r[f"gap_{k}"] = mv[k] - point[ref][k]
                    r[f"gap_{k}_lo"] = np.percentile(gap, 2.5) if len(gap) else np.nan
                    r[f"gap_{k}_hi"] = np.percentile(gap, 97.5) if len(gap) else np.nan
                    r[f"gap_{k}_p"] = float(min(1, 2 * min(np.mean(gap <= 0), np.mean(gap >= 0)))) if len(gap) else np.nan
            rows.append(r)
    t = pd.DataFrame(rows)
    pcols = [c for c in t.columns if c.endswith("_p") and c.startswith("gap_")]
    allp = t[pcols].values.ravel()
    q = bh_qvalues(allp).reshape(t[pcols].shape)
    for j, c in enumerate(pcols):
        t[c.replace("_p", "_q")] = q[:, j]
    small = t["n"] < 5
    t.loc[small, [c for c in t.columns if c not in ("task", "model", "weighting", "threshold", "group", "level",
                                                     "reference", "n")]] = np.nan
    return t


In [ ]:
%%writefile {SRC_DIR}/rtw_analyses.py
"""Analyses for revision v2 (writes aggregated CSV tables only)."""
from __future__ import annotations

import os

import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from scipy.stats import spearmanr, weightedtau

from rtw_config import (B_BOOT, B_SHAP, DCA_THRESHOLDS, FAIR_THRESHOLD, N_JOBS, NOM_LEVELS, PREDICTORS,
                        RESTRICTED_DROP, RESTRICTED_DROP_B, SEED, VAR_LABELS)
from rtw_data import design_matrix, small_vars, task_frame
from rtw_develop import designs_for
from rtw_metrics import (EPS, all_metrics, boot_indices, cal_slope, calibration_band, citl, dca_curves,
                         dca_vs_best_default, evaluate, logloss)
from rtw_models import ONEHOT_MODELS, aggregate_attr, attributions, fit, make_folds, oof_predict

QUICK = ("AUROC", "Brier", "LogLoss", "CITL", "Slope", "ICI")


def T(out, name):
    return os.path.join(out, "tables", name)


# =============================================================================================
# Descriptives and sample size
# =============================================================================================

def descriptives(a, out):
    s = a[a["part2"] == 1]
    rows = []

    def add(label, fn, kind="pct"):
        r = {"characteristic": label, "kind": kind}
        for c in (1, 2, 3):
            d = s[s["cohort"] == c]
            v = np.asarray(fn(d), float)
            w = d["wt_w2"].values
            ok = ~np.isnan(v)
            r[f"C{c}_unw"] = (100 if kind == "pct" else 1) * np.nanmean(v)
            r[f"C{c}_w"] = (100 if kind == "pct" else 1) * np.sum(w[ok] * v[ok]) / np.sum(w[ok])
        rows.append(r)
    for c in (1, 2, 3):
        pass
    n1 = {"characteristic": "Wave-1 participants, n", "kind": "n"}
    n2 = {"characteristic": "Analysed (responded at wave 2), n", "kind": "n"}
    rr = {"characteristic": "Wave-2 response, %", "kind": "pct_only"}
    for c in (1, 2, 3):
        n1[f"C{c}_unw"] = int((a["cohort"] == c).sum())
        n2[f"C{c}_unw"] = int(((a["cohort"] == c) & (a["part2"] == 1)).sum())
        rr[f"C{c}_unw"] = 100 * n2[f"C{c}_unw"] / n1[f"C{c}_unw"]
    rows += [n1, n2, rr]
    add("Female", lambda d: d["female"])
    add("Age >=60", lambda d: d["age_grp"] == 4)
    add("College or higher education", lambda d: d["edu"] == 5)
    add("Occupational disease", lambda d: d["disease"])
    add("Commuting injury", lambda d: d["commuting"])
    add("Disability grade 1-7", lambda d: d["disab_cat"].isin([1, 2]))
    add("No disability grade", lambda d: d["disab_cat"] == 6)
    add("Vocational/employment-support service", lambda d: d["svc_job"])
    add("Regular job at injury", lambda d: d["job_status"] == 1)
    add("Work ability, mean (0-10)", lambda d: d["work_ability"], kind="mean")
    add("High pain interference", lambda d: (d["pain_interf"] >= 3).astype(float).where(d["pain_interf"].notna()))
    add("Physical limitation", lambda d: d["phys_limitation"])
    add("Chronic disease", lambda d: d["chronic"])
    add("Employed at wave 1", lambda d: d["employed"])
    add("Employed at wave 2", lambda d: d["y"])
    add("A1: retained among employed at wave 1", lambda d: d["y"].where(d["employed"] == 1))
    add("A2: re-entered among not employed at wave 1", lambda d: d["y"].where(d["employed"] == 0))
    t = pd.DataFrame(rows)
    t.to_csv(T(out, "table1_characteristics.csv"), index=False)
    # task counts
    cnt = []
    for task in ("A1", "A2", "P"):
        d = task_frame(a, task)
        for c in (1, 2, 3):
            x = d[d.cohort == c]
            cnt.append({"task": task, "cohort": c, "n": len(x), "employed_w2": int(x["y"].sum()),
                        "undesired_events": int((1 - x["y"]).sum())})
    pd.DataFrame(cnt).to_csv(T(out, "task_counts.csv"), index=False)
    return t


def riley(res_by_task, a, out):
    """Riley et al. (2020) development sample-size criteria with R2_CS anticipated from the OOF log-likelihood
    of the unpenalised LR; expected shrinkage at the actual n; validation precision (O/E)."""
    rows = []
    for task, res in res_by_task.items():
        yd = res["y_dev"]
        n = len(yd)
        phi = yd.mean()
        P = len(res["columns"]["num"])
        def ll(y, p):
            p = np.clip(p, 1e-9, 1 - 1e-9)
            return np.sum(y * np.log(p) + (1 - y) * np.log(1 - p))
        l0 = ll(yd, np.full(n, phi))
        l1 = ll(yd, res["oof"]["LR"])
        r2cs = max(1 - np.exp(-2 * (l1 - l0) / n), 1e-6)
        maxr2 = 1 - np.exp(2 * l0 / n)
        n1 = P / ((0.9 - 1) * np.log(1 - r2cs / 0.9))
        s2 = r2cs / (r2cs + 0.05 * maxr2)
        n2 = P / ((s2 - 1) * np.log(1 - r2cs / s2))
        n3 = (1.96 / 0.05) ** 2 * phi * (1 - phi)
        shrink = 1 + P / (n * np.log(1 - r2cs))
        yt = res["y_test"]
        ph = yt.mean()
        se = np.sqrt((1 - ph) / (len(yt) * ph))
        rows.append({"task": task, "n_dev": n, "events_min_class": int(min(yd.sum(), n - yd.sum())),
                     "parameters": P, "EPP": min(yd.sum(), n - yd.sum()) / P, "R2cs_anticipated": r2cs,
                     "max_R2cs": maxr2, "n_crit1_shrinkage0.9": n1, "n_crit2_optimism": n2, "n_crit3_intercept": n3,
                     "n_required": max(n1, n2, n3), "expected_shrinkage_at_n": shrink,
                     "lr_oof_slope": cal_slope(yd, res["oof"]["LR"]), "n_test": len(yt),
                     "test_events_min": int(min(yt.sum(), len(yt) - yt.sum())),
                     "OE_95CI_lower_factor": np.exp(-1.96 * se), "OE_95CI_upper_factor": np.exp(1.96 * se)})
    t = pd.DataFrame(rows)
    t.to_csv(T(out, "sample_size_riley.csv"), index=False)
    return t


def predictor_table(a, out):
    s = a[a["part2"] == 1]
    rows = []
    for v in PREDICTORS:
        enc = ("one-hot" if v in NOM_LEVELS else "employed + nested emp6 indicators" if v in ("employed", "emp6")
               else "score (LR, trees); one-hot in ridge-OH")
        r = {"variable": v, "label": VAR_LABELS.get(v, v), "encoding": enc,
             "S4": "dropped" if v in RESTRICTED_DROP else "kept",
             "S4b": "dropped" if v in RESTRICTED_DROP_B else "kept"}
        for c in (1, 2, 3):
            r[f"missing_C{c}_%"] = round(100 * s.loc[s["cohort"] == c, v].isna().mean(), 1)
        rows.append(r)
    t = pd.DataFrame(rows)
    t.to_csv(T(out, "tableS1_predictors.csv"), index=False)
    return t


# =============================================================================================
# Temporal validation (primary design)
# =============================================================================================

def _ordered(res):
    order = ["Null", "Status", "LR-small", "LR", "RidgeOH", "RF", "XGB", "LGBM", "CatBoost", "EBM"]
    return {m: res["test"][m] for m in order if m in res["test"]}


def evaluate_task(res, a, out, B=B_BOOT, log=print):
    task = res["task"]
    d = task_frame(a, task)
    te = d.loc[res["test_index"]]
    yt = res["y_test"]
    preds = _ordered(res)
    refs = ("RidgeOH", "Status" if task == "P" else "LR-small")
    tabs = []
    for lab, w in (("unweighted", None), ("weighted", te["wt_w2"].values)):
        s, dd = evaluate(yt, preds, task, B=B, seed=SEED, w=w, strata=te["strata"].values, refs=refs, label=lab)
        s["task"] = task
        dd["task"] = task
        s.to_csv(T(out, f"perf_C3_{task}_{lab}.csv"), index=False)
        dd.to_csv(T(out, f"perf_C3_diffs_{task}_{lab}.csv"), index=False)
        tabs.append(s)
    dev = []
    for m, p in res["oof"].items():
        r = all_metrics(res["y_dev"], p, task)
        r.update({"model": m, "task": task, "cv_logloss": res["cv_logloss"].get(m, np.nan),
                  "repeated_cv_logloss": res["rep_logloss"].get(m, np.nan)})
        dev.append(r)
    pd.DataFrame(dev).to_csv(T(out, f"perf_devOOF_{task}.csv"), index=False)
    # slope / CITL change from development OOF to C3 (independent bootstraps of both samples)
    rng = np.random.default_rng(SEED)
    rows = []
    yd = res["y_dev"]
    bd = [rng.integers(0, len(yd), len(yd)) for _ in range(B)]
    bt = boot_indices(len(yt), B, SEED + 1, te["strata"].values)
    for m in preds:
        if m == "Null":
            continue
        po, pt = np.clip(res["oof"][m], EPS, 1 - EPS), np.clip(preds[m], EPS, 1 - EPS)
        for k, f in (("Slope", cal_slope), ("CITL", citl)):
            dv, tv = f(yd, po), f(yt, pt)
            s = np.array([f(yt[j], pt[j]) - f(yd[i], po[i]) for i, j in zip(bd, bt)], float)
            s = s[np.isfinite(s)]
            rows.append({"task": task, "model": m, "metric": k, "dev_oof": dv, "C3": tv, "change": tv - dv,
                         "lo": np.percentile(s, 2.5), "hi": np.percentile(s, 97.5)})
    pd.DataFrame(rows).to_csv(T(out, f"calibration_change_{task}.csv"), index=False)
    # decision curves
    thr = np.round(np.arange(0.02, 0.901, 0.01), 2)
    rules = {"Status rule": res["status_flag_test"]} if task == "P" else None
    keep = {m: preds[m] for m in (res["primary"], "RidgeOH", "LR-small", "Status") if m in preds}
    for lab, w in (("unweighted", None), ("weighted", te["wt_w2"].values)):
        dca_curves(yt, keep, thr, rules, w).assign(task=task, weighting=lab).to_csv(
            T(out, f"dca_{task}_{lab}.csv"), index=False)
        rows = []
        for m in (res["primary"], "RidgeOH"):
            dd = dca_vs_best_default(yt, preds[m], DCA_THRESHOLDS[task], rules, w, te["strata"].values, B=B, seed=SEED)
            dd["model"], dd["task"], dd["weighting"] = m, task, lab
            rows.append(dd)
        pd.concat(rows).to_csv(T(out, f"dca_delta_{task}_{lab}.csv"), index=False)
    # calibration curves with bands
    cal = []
    for m in dict.fromkeys([res["primary"], "RidgeOH", "LR-small"]):
        for lab, w in (("unweighted", None), ("weighted", te["wt_w2"].values)):
            cb = calibration_band(yt, preds[m], w, B=200, seed=SEED, strata=te["strata"].values)
            cb["model"], cb["weighting"], cb["task"] = m, lab, task
            cal.append(cb)
    pd.concat(cal).to_csv(T(out, f"calcurve_{task}.csv"), index=False)
    bins = np.linspace(0, 1, 41)
    hist = []
    for m in (res["primary"], "RidgeOH"):
        h, _ = np.histogram(preds[m], bins=bins)
        # 2026-10-15: small-cell suppression (integrity rule: no cell with 1-4 workers in any output table)
        small = (h > 0) & (h < 5)
        hist.append(pd.DataFrame({"model": m, "bin_lo": bins[:-1], "bin_hi": bins[1:],
                                  "count": pd.Series(h).mask(small).astype("Int64"), "suppressed_lt5": small}))
    pd.concat(hist).assign(task=task).to_csv(T(out, f"predhist_{task}.csv"), index=False)
    from sklearn.metrics import roc_curve
    roc = []
    grid = np.linspace(0, 1, 201)
    for m, p in preds.items():
        fpr, tpr, _ = roc_curve(yt, p)
        roc.append(pd.DataFrame({"model": m, "fpr": grid, "tpr": np.interp(grid, fpr, tpr)}))
    pd.concat(roc).assign(task=task).to_csv(T(out, f"roc_{task}.csv"), index=False)
    log(f"[evaluate {task}] done; primary {res['primary']}")


# =============================================================================================
# Secondary designs
# =============================================================================================

def quick_eval(y, preds, task, B, label, strata=None, w=None, extra=None):
    s, _ = evaluate(y, preds, task, B=B, seed=SEED, w=w, strata=strata, refs=(), which=QUICK, label="unweighted")
    s["analysis"], s["task"], s["n"], s["events"] = label, task, len(y), int(np.sum(y))
    for k, v in (extra or {}).items():
        s[k] = v
    return s


# =============================================================================================
# Attribution stability (supplement)
# =============================================================================================

def _imp(est, X, groups, max_rows=None):
    return aggregate_attr(attributions(est, X, max_rows=max_rows), groups)


def _split(r, model, params, Xd, yd, Xt, yt, groups, vars_):
    rng = np.random.default_rng(SEED + 9000 + r)
    h = len(yt) // 2
    it, idv = rng.permutation(len(yt)), rng.permutation(len(yd))
    parts = {"C3a": (Xt, yt, it[:h]), "C3b": (Xt, yt, it[h:2 * h]), "Da": (Xd, yd, idv[:h]), "Db": (Xd, yd, idv[h:2 * h])}
    imp = {}
    for k, (X, y, idx) in parts.items():
        est = fit(model, params, X.iloc[idx], y[idx], n_jobs=1)
        imp[k] = _imp(est, X.iloc[idx], groups).reindex(vars_).fillna(0).values
    rho = lambda u, v: spearmanr(imp[u], imp[v]).statistic  # noqa: E731
    wt = lambda u, v: weightedtau(imp[u], imp[v]).statistic  # noqa: E731
    return {"rep": r, "rho_within_C3": rho("C3a", "C3b"), "rho_within_dev": rho("Da", "Db"),
            "rho_between": np.mean([rho("Da", "C3a"), rho("Db", "C3b")]),
            "wtau_within_C3": wt("C3a", "C3b"), "wtau_within_dev": wt("Da", "Db"),
            "wtau_between": np.mean([wt("Da", "C3a"), wt("Db", "C3b")])}


def attribution_stability(res, a, out, R=30, log=print):
    task = res["task"]
    model = res["attr_model"]
    params = res["params"][model]
    d = task_frame(a, task)
    D = designs_for(d, task)
    X, groups = D["oh" if model in ONEHOT_MODELS else "num"]
    Xd, Xt = X.loc[res["dev_index"]], X.loc[res["test_index"]]
    yd, yt = res["y_dev"], res["y_test"]
    est = res["estimators"][model]
    svd = attributions(est, Xd)
    svt = attributions(est, Xt)
    vars_ = sorted(set(groups.values()))
    imp_d = aggregate_attr(svd, groups).reindex(vars_).fillna(0)
    imp_t = aggregate_attr(svt, groups).reindex(vars_).fillna(0)
    # per-feature difference (deployed model on C3 minus on dev), row bootstrap
    gmap = {c: groups.get(c.replace("missingindicator_", ""), c) for c in svd.columns}
    Ad = svd.T.groupby(pd.Series(gmap)).sum().T.abs().reindex(columns=vars_).fillna(0)
    At = svt.T.groupby(pd.Series(gmap)).sum().T.abs().reindex(columns=vars_).fillna(0)
    rng = np.random.default_rng(SEED)
    diffs = np.array([At.values[rng.integers(0, len(At), len(At))].mean(0) - Ad.values[rng.integers(0, len(Ad), len(Ad))].mean(0)
                      for _ in range(200)])
    est_c3 = fit(model, params, Xt, yt, n_jobs=N_JOBS)
    imp_r = _imp(est_c3, Xt, groups).reindex(vars_).fillna(0)
    tab = pd.DataFrame({"variable": vars_, "label": [VAR_LABELS.get(v, v) for v in vars_],
                        "deployed_on_dev": imp_d.values, "deployed_on_C3": imp_t.values,
                        "diff_C3_minus_dev": imp_t.values - imp_d.values,
                        "diff_lo": np.percentile(diffs, 2.5, axis=0), "diff_hi": np.percentile(diffs, 97.5, axis=0),
                        "refit_on_C3": imp_r.values}).sort_values("deployed_on_dev", ascending=False)
    tab["task"], tab["model"] = task, model
    tab.to_csv(T(out, f"attribution_{task}.csv"), index=False)
    sh = pd.DataFrame(Parallel(n_jobs=N_JOBS)(delayed(_split)(r, model, params, Xd, yd, Xt, yt, groups, vars_)
                                              for r in range(R)))
    top = lambda s: set(s.sort_values(ascending=False).index[:10])  # noqa: E731
    summ = {"task": task, "model": model,
            "rho_deployed_dev_vs_C3": spearmanr(imp_d, imp_t).statistic,
            "wtau_deployed_dev_vs_C3": weightedtau(imp_d.values, imp_t.values).statistic,
            "rho_dev_vs_C3_refit": spearmanr(imp_d, imp_r).statistic,
            "wtau_dev_vs_C3_refit": weightedtau(imp_d.values, imp_r.values).statistic,
            "top10_overlap_refit": len(top(imp_d) & top(imp_r)), "split_R": R}
    for c in sh.columns:
        if c != "rep":
            summ[f"{c}_median"] = sh[c].median()
            summ[f"{c}_p2_5"] = sh[c].quantile(0.025)
            summ[f"{c}_p97_5"] = sh[c].quantile(0.975)
    pd.DataFrame([summ]).to_csv(T(out, f"attribution_stability_{task}.csv"), index=False)
    log(f"[attribution {task}] {model}: rho deployed {summ['rho_deployed_dev_vs_C3']:.3f}, refit {summ['rho_dev_vs_C3_refit']:.3f}")
    return tab, summ


# =============================================================================================
# Sensitivity analyses
# =============================================================================================

def ipcw_weights(a):
    X, _ = design_matrix(a, "P")
    from rtw_models import make_estimator
    w = pd.Series(np.nan, index=a.index)
    for c in (1, 2, 3):
        m = (a["cohort"] == c).values
        z = a.loc[m, "part2"].values
        p = make_estimator("LR").fit(X[m], z).predict_proba(X[m])[:, 1]
        sw = z.mean() / p
        w[m] = np.minimum(sw, np.percentile(sw, 99))
    return w


def sensitivity(res, a, out, B=B_BOOT, log=print):
    task = res["task"]
    prim = res["primary"]
    models = list(dict.fromkeys([prim, "RidgeOH"]))
    d = task_frame(a, task)
    D = designs_for(d, task)
    dev, te = d.loc[res["dev_index"]], d.loc[res["test_index"]]
    yd, yt = res["y_dev"], res["y_test"]
    st = te["strata"].values
    rows = [quick_eval(yt, {m: res["test"][m] for m in models}, task, B, "Primary analysis", st)]

    def refit(m, Xsrc, w=None):
        X = Xsrc
        return fit(m, res["params"][m], X.loc[dev.index], yd, n_jobs=N_JOBS, sample_weight=w).predict_proba(
            X.loc[te.index])[:, 1]
    Xof = lambda m, D_=D: D_["oh" if m in ONEHOT_MODELS else "num"][0]  # noqa: E731
    wdev = (dev["wt_w2"] / dev.groupby("cohort")["wt_w2"].transform("mean")).values
    p = {m: refit(m, Xof(m), wdev) for m in models}
    rows.append(quick_eval(yt, p, task, B, "Survey-weighted training, weighted evaluation", st, te["wt_w2"].values))
    ip = ipcw_weights(a)
    p = {m: refit(m, Xof(m), ip.reindex(dev.index).values) for m in models}
    # inverse probability of response weighting (IPRW); the CSV label is kept for the manuscript placeholders
    rows.append(quick_eval(yt, p, task, B, "IPCW training and evaluation", st, ip.reindex(te.index).values))
    for lab, drop in (("Restricted predictors (S4)", RESTRICTED_DROP), ("Restricted predictors (S4b)", RESTRICTED_DROP_B),
                      ("Work ability removed", ["work_ability"])):
        keepv = [v for v in PREDICTORS if v not in drop]
        D2 = designs_for(d, task, keepv)
        p = {m: refit(m, Xof(m, D2)) for m in models}
        rows.append(quick_eval(yt, p, task, B, lab, st))
    m6 = (te["commuting"] != 1).values
    rows.append(quick_eval(yt[m6], {m: res["test"][m][m6] for m in models}, task, B,
                           "C3 excluding commuting injuries", st[m6]))
    t = pd.concat(rows, ignore_index=True)
    t.to_csv(T(out, f"sensitivity_{task}.csv"), index=False)
    log(f"[sensitivity {task}] done")
    return t


def seed_variability(res, a, out, n_seeds=20):
    task = res["task"]
    m = res["primary"]
    if m in ("LR", "RidgeOH"):
        return None
    d = task_frame(a, task)
    X = designs_for(d, task)["oh" if m in ONEHOT_MODELS else "num"][0]
    Xd, Xt = X.loc[res["dev_index"]], X.loc[res["test_index"]]
    yt = res["y_test"]
    rows = []
    for s in range(n_seeds):
        p = fit(m, res["params"][m], Xd, res["y_dev"], n_jobs=N_JOBS, seed=SEED + 500 + s).predict_proba(Xt)[:, 1]
        r = all_metrics(yt, p, task, which=("AUROC", "Brier", "CITL", "Slope"))
        r["seed"] = s
        rows.append(r)
    t = pd.DataFrame(rows)
    t["task"], t["model"] = task, m
    t.to_csv(T(out, f"seed_variability_{task}.csv"), index=False)
    return t


# =============================================================================================
# Extras: within-stratum performance of the pooled models; attrition
# =============================================================================================

def within_strata(res_p, a, out, B=B_BOOT):
    """Pooled-task models evaluated within wave-1 employment strata (employed / not employed)."""
    d = task_frame(a, "P")
    te = d.loc[res_p["test_index"]]
    yt = res_p["y_test"]
    rows = []
    for e, lab in ((1, "employed at wave 1"), (0, "not employed at wave 1")):
        m = (te["employed"].values == e)
        preds = {k: v[m] for k, v in _ordered(res_p).items() if k not in ("Null",)}
        for wl, w in (("unweighted", None), ("weighted", te["wt_w2"].values[m])):
            s, _ = evaluate(yt[m], preds, "P", B=B, seed=SEED, w=w, strata=te["strata"].values[m], refs=(),
                            which=("AUROC", "CITL", "Slope", "ICI"), label=wl)
            s["stratum"], s["n"], s["employed_w2"] = lab, int(m.sum()), int(yt[m].sum())
            rows.append(s)
    t = pd.concat(rows, ignore_index=True)
    t.to_csv(T(out, "perf_C3_P_within_strata.csv"), index=False)
    return t


def attrition(a, out):
    """Wave-1 to wave-2 response by cohort and wave-1 employment; missing wave-2 weights among respondents."""
    rows = []
    for c in (1, 2, 3):
        d = a[a["cohort"] == c]
        r = {"cohort": c, "wave1_n": len(d), "wave2_n": int(d["part2"].sum()),
             "response_pct": 100 * d["part2"].mean(),
             "response_pct_employed_w1": 100 * d.loc[d["employed"] == 1, "part2"].mean(),
             "response_pct_not_employed_w1": 100 * d.loc[d["employed"] == 0, "part2"].mean(),
             "missing_wt_w2_among_respondents": int(d.loc[d["part2"] == 1, "wt_w2"].isna().sum()),
             "missing_outcome_among_respondents": int(d.loc[d["part2"] == 1, "y"].isna().sum())}
        rows.append(r)
    t = pd.DataFrame(rows)
    t.to_csv(T(out, "attrition.csv"), index=False)
    return t


def weights_design(a, out):
    """Round 2: survey-weight diagnostics per cohort and task (analysed wave-2 respondents): Kish effective sample size
    of the wave-2 longitudinal weights, max/min weight ratio, number of sampling strata and of strata with a single
    analysed unit (these make the within-stratum bootstrap mildly anti-conservative)."""
    rows = []
    for task in ("A1", "A2", "P"):
        d = task_frame(a, task)
        for c in (1, 2, 3):
            g = d[d["cohort"] == c]
            w = g["wt_w2"].values.astype(float)
            has_strata = g["strata"].notna().any()      # sampling strata are available for the evaluation cohort
            sizes = g.groupby("strata").size()
            rows.append({"task": task, "cohort": c, "n": len(g), "kish_ess": w.sum() ** 2 / np.sum(w ** 2),
                         "ess_pct": 100 * (w.sum() ** 2 / np.sum(w ** 2)) / len(g), "max_min_weight_ratio": w.max() / w.min(),
                         "n_strata": int(len(sizes)) if has_strata else np.nan,
                         "single_unit_strata": int((sizes == 1).sum()) if has_strata else np.nan})
    t = pd.DataFrame(rows)
    t.to_csv(T(out, "weights_design.csv"), index=False)
    return t


def disde_restricted(res, sec, a, out, log=print):
    """Round 2 robustness: prevalence DISDE decomposition (model-free) with the domain classifier restricted to the
    comparability-restricted predictor set S4b, for the pooled and cohort-specific designs."""
    import rtw_shift as SH
    from rtw_develop import designs_for
    variables = [v for v in PREDICTORS if v not in RESTRICTED_DROP_B]
    rows = []
    for task in ("A1", "A2", "P"):
        d = task_frame(a, task)
        Xn = designs_for(d, task, variables=variables)["num"][0]
        for lab, r in (("C1+C2 -> C3", res[task]), ("C1 -> C3", sec[("c1to3", task)]),
                       ("C2 -> C3", sec[("c2to3", task)])):
            Xs, Xt = Xn.loc[r["dev_index"]], Xn.loc[r["test_index"]]
            tab, _ = SH.disde(Xs, r["y_dev"], {"RidgeOH": r["oof"]["RidgeOH"]}, Xt, r["y_test"],
                              {"RidgeOH": r["test"]["RidgeOH"]})
            tab = tab[tab.metric == "Prevalence"].copy()
            tab["task"], tab["design"], tab["predictors"] = task, lab, "S4b (restricted)"
            rows.append(tab)
        log(f"[disde restricted {task}] done")
    t = pd.concat(rows, ignore_index=True)
    t.to_csv(T(out, "shift_disde_restricted.csv"), index=False)
    return t


In [ ]:
%%writefile {SRC_DIR}/rtw_figures.py
"""Figures for revision v2, drawn only from aggregated CSV tables (greyscale-readable).

Layout pass 2026-10-13 (no change to what is plotted): legends that sat on marks were moved outside the axes
(_legend_outside) or into a corner verified to be empty, the "all (n)" x tick label of Fig. 4b-c is staggered so
it cannot collide with "1000", the smallest text was raised to 6 pt, and the within-C3 series got distinct markers.
Second pass (same day): the Fig. S1 legend is now a figure legend built from the Pooled panel, which carries all six
strategies (the in-panel legend of A1 lacked "Status rule"), "Flag none" got a dash-dot linestyle so that it is not
colour-only versus "Parsimonious LR", and the Fig. 2 calibration legends were tightened so they clear the CI bands.
Consistency corrections 2026-10-15 (no analysis re-run; drawn from the same CSVs): Fig. 1 now follows the steps of
Algorithm 1 (label-free step before outcomes arrive; no re-development loop); estimands are labelled "sample" and
"population" as in the text; model names follow the tables (Random forest, Status-only LR, LR (scores)); Fig. 3a axis
is a log odds ratio and Fig. 3b is the DISDE-type decomposition (C1+C2 tick); Fig. 4a shows CatBoost only (the C1 -> C2
rows, silently XGBoost, were dropped) on a log axis with an outside legend, and the Fig. 4b-c whiskers are dodged by
method; the subgroup levels are ordered consistently with typographic signs; Fig. S4 uses the Fig. 4 styles and has an
x label and a "No update" legend entry; a zero attribution in Fig. S5 is drawn at the axis floor instead of dropped.
"""
from __future__ import annotations

import os

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402
import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
from matplotlib.patches import FancyBboxPatch  # noqa: E402

from rtw_config import MODEL_LABEL, VAR_LABELS  # noqa: E402

plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 7.5, "axes.titlesize": 7.8, "axes.labelsize": 7.3,
    "legend.fontsize": 6.3, "xtick.labelsize": 6.5, "ytick.labelsize": 6.5, "axes.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False, "savefig.dpi": 300,
    "axes.grid": True, "grid.color": "#e6e6e6", "grid.linewidth": 0.4,
})
GREYS = ["#111111", "#555555", "#8a8a8a", "#b5b5b5"]
TASK_TITLE = {"A1": "A1 retention", "A2": "A2 re-entry", "P": "Pooled"}
DESIGNS = ["C1 -> C3", "C2 -> C3", "C1+C2 -> C3"]
SHORT = {"Null": "Null", "Status": "Status-only LR", "LR-small": "Parsimonious LR", "LR": "LR (scores)",
         "RidgeOH": "Ridge-OH", "RF": "Random forest", "XGB": "XGBoost", "LGBM": "LightGBM", "CatBoost": "CatBoost",
         "EBM": "EBM"}
ESTIMAND = {"unweighted": "sample", "weighted": "population"}
ESTIMAND_LONG = {"unweighted": "sample (unweighted)", "weighted": "population (weighted)"}
# display labels only (the CSV values stay as written by the analysis code)
LABELS = {"Age >=60": "Age \u226560", "Disability grade 1-7": "Disability grade 1\u20137",
          "Work ability, mean (0-10)": "Work ability, mean (0\u201310)", "<=39": "\u226439", ">=60": "\u226560",
          "40-49": "40\u201349", "50-59": "50\u201359", "Mild (8-14)": "Mild (8\u201314)",
          "Severe (1-7)": "Severe (1\u20137)", "Work ability (0-10)": "Work ability (0\u201310)"}


def _lab(x):
    return LABELS.get(x, x)


def _save(fig, out, name):
    d = os.path.join(out, "figures")
    fig.savefig(os.path.join(d, name + ".png"), dpi=300, bbox_inches="tight")
    fig.savefig(os.path.join(d, name + ".pdf"), bbox_inches="tight")
    plt.close(fig)


def _legend_outside(fig, axes_span, handles, labels, ncol, side="below", fontsize=6.0, gap=0.006):
    """Figure legend centred on the given axes and placed entirely outside the figure area (below or above), so it
    can never cover marks, tick labels or axis labels; bbox_inches="tight" in _save extends the saved page to
    include it. Call after fig.tight_layout() so that the axes positions are final."""
    x0 = min(a.get_position().x0 for a in axes_span)
    x1 = max(a.get_position().x1 for a in axes_span)
    if side == "below":
        anchor, loc = ((x0 + x1) / 2, -gap), "upper center"
    else:
        anchor, loc = ((x0 + x1) / 2, 1 + gap), "lower center"
    return fig.legend(handles, labels, loc=loc, bbox_to_anchor=anchor, ncol=ncol, frameon=False, fontsize=fontsize,
                      handlelength=1.8, columnspacing=1.2, handletextpad=0.5)


def _t(out, name):
    return pd.read_csv(os.path.join(out, "tables", name))


def _primary(out, task):
    t = _t(out, "tuning_primary.csv")
    t = t[(t.task == task)]
    return t.loc[t.repeated_cv_logloss.idxmin(), "model"]


def fig1_framework(out):
    """Protocol figure; the boxes follow the steps of Algorithm 1 in the manuscript (2026-10-15)."""
    fig, ax = plt.subplots(figsize=(6.9, 2.2))
    ax.axis("off")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    steps = [("Harmonise", "C1\u2013C3\n(wave 1 in 2013,\n2018 and 2023)"),
             ("Step 1: Develop", "C1+C2; seven\nlearners; selection\nby repeated cross-\nvalidation; keep\nout-of-fold\npredictions"),
             ("Step 2: New covariates\n(no outcomes yet)", "adversarial AUROC;\ndensity ratio and effective\n"
              "sample size; importance-\nweighted recalibration only if\nthe shift is mainly in X and the\n"
              "weighted out-of-fold calibration\nimproves; otherwise keep the\nmodel and monitor"),
             ("Step 3a\u2013b:\nOutcomes arrive", "evaluate in the\nsample and the\npopulation; decompose\nthe change by\nsource cohort"),
             ("Step 3c\u2013d:\nUpdate and audit", "recalibrate with\nrecent records when\nmiscalibration is\nclear; audit\nsubgroup errors")]
    widths = [0.13, 0.175, 0.245, 0.18, 0.18]
    gap, y0, h = 0.019, 0.25, 0.73
    xs = []
    x = 0.008
    for i, ((title, body), w) in enumerate(zip(steps, widths)):
        xs.append((x, w))
        ax.add_patch(FancyBboxPatch((x, y0), w, h, boxstyle="round,pad=0.005,rounding_size=0.02",
                                    fc="#f0f0f0" if i % 2 == 0 else "white", ec="#222222", lw=0.8))
        ax.text(x + w / 2, y0 + h - 0.05, title, ha="center", va="top", fontsize=6.3, fontweight="bold",
                linespacing=1.15)
        ax.text(x + w / 2, y0 + h - 0.25, body, ha="center", va="top", fontsize=5.6, linespacing=1.25)
        if i < len(steps) - 1:
            ax.annotate("", xy=(x + w + gap, y0 + h / 2), xytext=(x + w, y0 + h / 2),
                        arrowprops=dict(arrowstyle="-|>", lw=0.9, color="#222222", mutation_scale=7,
                                        shrinkA=0, shrinkB=0))
        x += w + gap
    x_from = xs[4][0] + xs[4][1] / 2
    x_to = xs[2][0] + xs[2][1] / 2
    yl = 0.12
    ax.plot([x_from, x_from, x_to], [y0 - 0.01, yl, yl], color="#777777", lw=0.7, ls="--")
    ax.annotate("", xy=(x_to, y0 - 0.01), xytext=(x_to, yl),
                arrowprops=dict(arrowstyle="-|>", lw=0.7, color="#777777", ls="--", shrinkA=0, shrinkB=0))
    ax.text((x_from + x_to) / 2, yl - 0.02, "next cohort: return to step 2", ha="center", va="top", fontsize=6.0,
            color="#555555")
    _save(fig, out, "fig1_framework")


def fig2_performance(out):
    """(a,b) calibration curves with bands, unweighted and weighted, for A1 and A2; (c) AUROC by model."""
    fig = plt.figure(figsize=(6.9, 4.9))
    gs = fig.add_gridspec(2, 3, width_ratios=[1, 1, 1.15])
    for j, task in enumerate(("A1", "A2")):
        cal = _t(out, f"calcurve_{task}.csv")
        prim = _primary(out, task)
        for i, lab in enumerate(("unweighted", "weighted")):
            ax = fig.add_subplot(gs[i, j])
            for m, col, ls in ((prim, "#111111", "-"), ("RidgeOH", "#777777", (0, (4, 2)))):
                d = cal[(cal.model == m) & (cal.weighting == lab)]
                if len(d) == 0:
                    continue
                ax.fill_between(d.pred, d.lo, d.hi, color=col, alpha=0.15, lw=0)
                ax.plot(d.pred, d.obs, color=col, ls=ls, lw=1.1, label=SHORT.get(m, m))
            lo = 0.4 if task == "A1" else 0
            ax.plot([lo, 1], [lo, 1], color="#bbbbbb", lw=0.6)
            ax.set_xlim(lo, 1)
            ax.set_ylim(lo, 1)
            ax.set_title(f"({'abcd'[2 * j + i]}) {TASK_TITLE[task]}, {ESTIMAND[lab]}", loc="left")
            ax.set_xlabel("Predicted P(employed)")
            ax.set_ylabel("Observed")
            # compact legend: with the default pad/handle length the "Ridge-OH" label touched the CI band in (b)
            ax.legend(frameon=False, loc="upper left", borderaxespad=0.2, handlelength=1.5)
    ax = fig.add_subplot(gs[:, 2])
    rows = []
    for task in ("A1", "A2", "P"):
        for lab in ("unweighted", "weighted"):
            t = _t(out, f"perf_C3_{task}_{lab}.csv")
            t = t[(t.metric == "AUROC") & (t.model != "Null")]
            for _, r in t.iterrows():
                rows.append((task, lab, r.model, r.estimate, r.lo, r.hi))
    df = pd.DataFrame(rows, columns=["task", "w", "model", "est", "lo", "hi"])
    order = ["Status", "LR-small", "LR", "RidgeOH", "RF", "XGB", "LGBM", "CatBoost", "EBM"]
    ylab, y = [], 0
    for task in ("A1", "A2", "P"):
        for m in order:
            dd = df[(df.task == task) & (df.model == m)]
            if len(dd) == 0:
                continue
            for lab, mk, fc, off in (("unweighted", "o", "#111111", 0.15), ("weighted", "s", "white", -0.15)):
                r = dd[dd.w == lab]
                if len(r):
                    r = r.iloc[0]
                    ax.errorbar(r.est, -y + off, xerr=np.clip(np.array([[r.est - r.lo], [r.hi - r.est]], dtype=float), 0, None), fmt=mk, ms=2.8, mfc=fc,
                                mec="#111111", ecolor="#666666", elinewidth=0.6, capsize=0,
                                label=ESTIMAND_LONG[lab] if (task == "A1" and m == "LR-small") else None)
            ylab.append((-y, f"{task}: {SHORT[m]}"))
            y += 1
        y += 0.6
    ax.set_yticks([p for p, _ in ylab])
    ax.set_yticklabels([l for _, l in ylab], fontsize=6.0)
    ax.set_xlabel("AUROC in C3 (95% CI)")
    ax.set_title("(e) Discrimination", loc="left")
    h_e, l_e = ax.get_legend_handles_labels()
    fig.tight_layout()
    # legend of (e) below the panel (2026-10-15): inside the axes the longer estimand labels touched the P rows
    _legend_outside(fig, [ax], h_e, l_e, ncol=1, side="below")
    _save(fig, out, "fig2_performance")


def fig3_shift(out):
    ce = _t(out, "shift_cohort_effects.csv")
    dis = _t(out, "shift_disde.csv")
    adv = _t(out, "shift_adversarial.csv")
    fig, axes = plt.subplots(1, 3, figsize=(6.9, 2.6), gridspec_kw={"width_ratios": [1, 1.5, 0.9]})
    ax = axes[0]
    y = 0
    labels = []
    for task in ("A1", "A2", "P"):
        for c in ("C2 vs C1", "C3 vs C1", "C3 vs C2"):
            r = ce[(ce.task == task) & (ce.contrast == c)].iloc[0]
            ax.errorbar(r.log_odds, -y, xerr=np.clip(np.array([[r.log_odds - r.lo], [r.hi - r.log_odds]], dtype=float), 0, None), fmt="o", ms=3, color="#111111",
                        elinewidth=0.7)
            labels.append((-y, f"{TASK_TITLE[task]}: {c}"))
            y += 1
        y += 0.5
    ax.axvline(0, color="#888888", lw=0.6)
    ax.set_yticks([p for p, _ in labels])
    ax.set_yticklabels([l for _, l in labels], fontsize=6.0)
    ax.set_xlabel("Adjusted log odds ratio of employment")
    ax.set_title("(a) Conditional cohort effects", loc="left")
    ax = axes[1]
    comps = [("X_shift_source_side", "X-shift (source)", "#d9d9d9", None), ("YgivenX_shift", "Y|X shift", "#444444", None),
             ("X_shift_target_side", "X-shift (target)", "white", "////")]
    xs = []
    k = 0
    for task in ("A1", "A2"):
        for des in DESIGNS:
            sub = dis[(dis.task == task) & (dis.design == des) & (dis.metric == "Prevalence")]
            if len(sub) == 0:
                continue
            bottom_pos, bottom_neg = 0, 0
            for comp, lab, fc, hatch in comps:
                r = sub[sub.component == comp].iloc[0]
                v = r.estimate
                b = bottom_pos if v >= 0 else bottom_neg
                ax.bar(k, v, bottom=b, color=fc, edgecolor="#111111", lw=0.5, hatch=hatch, width=0.7,
                       label=lab if k == 0 else None)
                if v >= 0:
                    bottom_pos += v
                else:
                    bottom_neg += v
            tot = sub[sub.component == "total"].iloc[0]
            ax.plot(k, tot.estimate, "D", ms=3.5, mfc="white", mec="#111111", label="Total change" if k == 0 else None)
            xs.append((k, {"C1 -> C3": "C1", "C2 -> C3": "C2", "C1+C2 -> C3": "C1+C2"}[des] + f"\n{task}"))
            k += 1
        k += 0.6
    ax.axhline(0, color="#888888", lw=0.6)
    ax.set_xticks([p for p, _ in xs])
    ax.set_xticklabels([l for _, l in xs], fontsize=6.0)
    ax.set_ylabel("Change in employment prevalence")
    ax.set_xlabel("Source cohort(s) → C3")
    ax.set_title("(b) DISDE-type decomposition", loc="left")
    h_b, l_b = ax.get_legend_handles_labels()   # legend drawn below the panel (an inside legend covered the C1/A2 bar)
    ax = axes[2]
    a = adv[adv.predictors == "all"].reset_index(drop=True)
    b = adv[adv.predictors == "restricted"].reset_index(drop=True)
    for i, r in a.iterrows():
        ax.errorbar(r.AUROC, -i + 0.12, xerr=np.clip(np.array([[r.AUROC - r.lo], [r.hi - r.AUROC]], dtype=float), 0, None), fmt="o", ms=3, color="#111111",
                    label="all predictors" if i == 0 else None)
    for i, r in b.iterrows():
        ax.errorbar(r.AUROC, -i - 0.12, xerr=np.clip(np.array([[r.AUROC - r.lo], [r.hi - r.AUROC]], dtype=float), 0, None), fmt="s", ms=3, mfc="white",
                    mec="#111111", ecolor="#111111", label="restricted (S4b)" if i == 0 else None)
    ax.set_yticks([-i for i in range(len(a))])
    ax.set_yticklabels(a.contrast.str.replace("+", "+", regex=False), fontsize=6.0)
    ax.set_xlabel("Domain-classifier AUROC")
    ax.set_title("(c) Adversarial validation", loc="left")
    h_c, l_c = ax.get_legend_handles_labels()   # legend drawn below the panel (an inside legend covered C1 vs C2)
    fig.tight_layout()
    _legend_outside(fig, [axes[1]], h_b, l_b, ncol=2, side="below")
    _legend_outside(fig, [axes[2]], h_c, l_c, ncol=1, side="below")
    _save(fig, out, "fig3_shift")


def fig4_updating(out):
    iw = _t(out, "update_iw_recalibration.csv")
    lg = _t(out, "update_lagged_summary.csv")
    fig, axes = plt.subplots(1, 3, figsize=(6.9, 2.5))
    ax = axes[0]
    meths = [("No update", "o", "#111111"), ("OOF recalibration (unweighted)", "^", "#888888"),
             ("IW logistic recalibration", "s", "white")]
    IW_LABEL = {"No update": "No update", "OOF recalibration (unweighted)": "Unweighted recal. (out-of-fold)",
                "IW logistic recalibration": "Importance-weighted recal."}
    ylab = []
    y = 0
    for task in ("A1", "A2"):
        prim = _primary(out, task)
        for des in DESIGNS:   # 2026-10-15: C3 designs only, primary model only (no silent XGBoost fallback)
            sub = iw[(iw.task == task) & (iw.design == des) & (iw.model == prim) & (iw.metric == "Slope")]
            if len(sub) == 0:
                raise ValueError(f"no {prim} rows for {task} {des} in update_iw_recalibration.csv")
            for j, (meth, mk, fc) in enumerate(meths):
                r = sub[sub.method == meth]
                if len(r):
                    r = r.iloc[0]
                    ax.errorbar(r.estimate, -y + (1 - j) * 0.18, xerr=np.clip(np.array([[r.estimate - r.lo], [r.hi - r.estimate]], dtype=float), 0, None),
                                fmt=mk, ms=2.8, mfc=fc, mec="#111111", ecolor="#777777", elinewidth=0.6,
                                label=IW_LABEL[meth] if y == 0 else None)
            ylab.append((-y, f"{task}: {des.replace(' -> ', '→')}"))
            y += 1
        y += 0.5
    ax.axvline(1, color="#888888", lw=0.6)
    ax.set_xscale("log")
    ax.set_xticks([0.5, 1, 2, 4, 8])
    ax.set_xticklabels(["0.5", "1", "2", "4", "8"])
    ax.minorticks_off()
    ax.set_yticks([p for p, _ in ylab])
    ax.set_yticklabels([l for _, l in ylab], fontsize=6.0)
    ax.set_xlabel("Calibration slope in C3 (log scale)")
    ax.set_title("(a) Label-free recalibration", loc="left")
    h_a, l_a = ax.get_legend_handles_labels()   # drawn below the panel: on a log axis an inside legend covered rows
    sty = {"Intercept recalibration": ("#111111", "-", "o"), "Logistic recalibration": ("#555555", (0, (4, 2)), "s"),
           "Closed testing": ("#888888", (0, (1, 1)), "^"), "Pooled refit (C1 + C2 sample)": ("#111111", (0, (5, 1, 1, 1)), "D"),
           "Ridge refit on C2 sample only": ("#aaaaaa", "-", "v")}
    for k, task in enumerate(("A1", "A2")):
        ax = axes[k + 1]
        s = lg[lg.task == task]
        base = s[s.method == "No update (C1 model)"].iloc[0]
        ax.axhline(base["ICI_median"], color="#bbbbbb", lw=1.2, label="No update (C1 model)")
        for mi, (meth, (c, ls, mk)) in enumerate(sty.items()):
            d = s[(s.method == meth)].sort_values("n")
            if len(d) == 0:
                continue
            x = np.log10(d.n) + (mi - 2) * 0.012   # dodge so that each method's whisker is visible
            ax.plot(x, d["ICI_median"], color=c, ls=ls, marker=mk, ms=2.8, lw=0.9, mfc="white", label=meth)
            ax.vlines(x, d["ICI_p2_5"], d["ICI_p97_5"], color=c, lw=0.5)
        ns = sorted(s[s.n > 0].n.unique())
        ticks = [n for n in ns if n in (100, 250, 500, 1000)] + [max(ns)]
        ax.set_xticks(np.log10(ticks))
        # "all (n)" is written one line lower than the numeric ticks: log10(1130) sits next to log10(1000)
        ax.set_xticklabels([str(int(n)) for n in ticks[:-1]] + [f"\nall\n({int(ticks[-1])})"], fontsize=6.0)
        ax.set_xlabel("C2 records used for updating")
        ax.set_ylabel("ICI in C3")
        ax.set_title(f"({'bc'[k]}) Lagged update, {TASK_TITLE[task]}", loc="left")
    fig.tight_layout()
    # shared legend of (b) and (c) under those panels; an inside legend covered the no-update and refit lines in (c)
    _legend_outside(fig, axes[1:], *axes[2].get_legend_handles_labels(), ncol=3, side="below")
    _legend_outside(fig, [axes[0]], h_a, l_a, ncol=1, side="below")
    _save(fig, out, "fig4_updating")


def figS_fairness(out):
    f = _t(out, "fairness.csv")
    for task in ("A1", "A2"):
        sub = f[(f.task == task) & (f.weighting == "unweighted")]
        prim = _primary(out, task)
        rows = sub[(sub.level != sub.reference) & sub.gap_TPR.notna()]
        order = rows[rows.model == prim][["group", "level"]].drop_duplicates().values.tolist()
        # 2026-10-15: keep the group order but sort the levels within a group (age was non-monotonic in Fig. S2)
        rank = {lv: k for k, lv in enumerate(["Female", "40-49", "50-59", ">=60", "Severe (1-7)", "No grade",
                                               "Disease", "Non-regular"])}
        order = sorted(order, key=lambda o: (order.index(next(p for p in order if p[0] == o[0])), rank.get(o[1], 99)))
        mets = [("AUROC", "Δ AUROC"), ("CITL", "Δ CITL"), ("TPR", "Δ TPR"), ("FPR", "Δ FPR")]
        fig, axes = plt.subplots(1, 4, figsize=(6.9, 2.6), sharey=True)
        yy = np.arange(len(order))[::-1]
        for i, (k, lab) in enumerate(mets):
            ax = axes[i]
            for model, off, mk, fc in ((prim, 0.12, "o", "#111111"), ("RidgeOH", -0.12, "s", "white")):
                d = rows[rows.model == model].set_index(["group", "level"])
                if len(d) == 0 or (model == "RidgeOH" and prim == "RidgeOH"):
                    continue
                e = np.array([d.loc[tuple(o), f"gap_{k}"] for o in order])
                lo = np.array([d.loc[tuple(o), f"gap_{k}_lo"] for o in order])
                hi = np.array([d.loc[tuple(o), f"gap_{k}_hi"] for o in order])
                ax.errorbar(e, yy + off, xerr=np.clip(np.array([e - lo, hi - e], dtype=float), 0, None), fmt=mk, ms=2.8, mfc=fc, mec="#111111",
                            ecolor="#666666", elinewidth=0.6, label=SHORT.get(model, model))
            ax.axvline(0, color="#888888", lw=0.6)
            ax.set_title(lab, loc="left")
            ax.grid(axis="y", visible=False)
        ref = rows.drop_duplicates("group").set_index("group")["reference"]
        axes[0].set_yticks(yy)
        axes[0].set_yticklabels([f"{g}: {_lab(lv)} vs {_lab(ref[g])}" for g, lv in order], fontsize=6.0)
        h, l = axes[0].get_legend_handles_labels()
        fig.tight_layout()
        _legend_outside(fig, axes, h, l, ncol=2, side="above", fontsize=6.3)
        _save(fig, out, f"figS_fairness_{task}")


def figS_dca(out):
    fig, axes = plt.subplots(2, 3, figsize=(6.9, 4.2))
    for j, task in enumerate(("A1", "A2", "P")):
        for i, lab in enumerate(("unweighted", "weighted")):
            ax = axes[i, j]
            d = _t(out, f"dca_{task}_{lab}.csv")
            # "Flag none" is dash-dot so that it differs from "Parsimonious LR" by more than its grey level
            styles = {"Flag all": ("#777777", (0, (2, 2))), "Flag none": ("#aaaaaa", (0, (3, 1, 1, 1))),
                      "Status rule": ("#555555", (0, (1, 1))), "RidgeOH": ("#777777", (0, (4, 2))),
                      "LR-small": ("#bbbbbb", "-")}
            prim = _primary(out, task)
            styles[prim] = ("#111111", "-")
            ymax = 0
            for s, (c, ls) in styles.items():
                dd = d[d.strategy == s]
                if len(dd):
                    ax.plot(dd.threshold, dd.net_benefit, color=c, ls=ls, lw=1.0, label=SHORT.get(s, s))
                    ymax = max(ymax, dd.net_benefit.max())
            ax.set_ylim(-0.02, ymax * 1.1 + 1e-3)
            ax.set_xlim(0.02, 0.9)
            ax.set_title(f"{TASK_TITLE[task]}, {ESTIMAND[lab]}", loc="left")
            ax.set_xlabel("Threshold risk of undesired outcome")
            ax.set_ylabel("Net benefit")
    fig.tight_layout()
    # one legend above all panels, taken from the Pooled-unweighted axes: only the pooled tables contain the
    # "Status rule" strategy, so a legend built from the A1 panel would miss that curve
    _legend_outside(fig, list(axes.flat), *axes[0, 2].get_legend_handles_labels(), ncol=6, side="above")
    _save(fig, out, "figS_decision_curves")


def figS_withinc3(out):
    s = _t(out, "update_withinC3_summary.csv")
    fig, axes = plt.subplots(2, 3, figsize=(6.9, 4.0))
    for i, task in enumerate(("A1", "A2")):
        d0 = s[s.task == task]
        base = d0[d0.method == "No update"].iloc[0]
        for j, k in enumerate(("AUROC", "Brier", "ICI")):
            ax = axes[i, j]
            ax.axhline(base[f"{k}_median"], color="#bbbbbb", lw=1.2, label="No update" if (i == 0 and j == 0) else None)
            # markers, grey levels and line styles follow Fig. 4 for the same methods (2026-10-15)
            for meth, c, ls, mk in (("Intercept recalibration", "#111111", "-", "o"),
                                    ("Logistic recalibration", "#555555", (0, (4, 2)), "s"),
                                    ("Closed testing", "#888888", (0, (1, 1)), "^"),
                                    ("Refit on C3 sample (development hyperparameters)", "#777777", (0, (2, 1)), "P"),
                                    ("Ridge refit on C3 sample (CV-tuned)", "#aaaaaa", "-", "v"),
                                    ("Pooled refit (dev + C3 sample)", "#111111", (0, (5, 1, 1, 1)), "D")):
                d = d0[d0.method == meth].sort_values("n")
                if len(d):
                    ax.plot(np.log10(d.n), d[f"{k}_median"], color=c, ls=ls, marker=mk, ms=2.6, mfc="white", lw=0.9,
                            label=meth if (i == 0 and j == 0) else None)
            ns = sorted(d0[d0.n > 0].n.unique())
            ax.set_xticks(np.log10(ns))
            ax.set_xticklabels([str(int(n)) for n in ns])
            ax.set_title(f"{TASK_TITLE[task]}: {k}", loc="left")
            if i == 1:
                ax.set_xlabel("C3 records used for updating")
    fig.tight_layout()
    _legend_outside(fig, list(axes.flat), *axes[0, 0].get_legend_handles_labels(), ncol=3, side="above")
    _save(fig, out, "figS_withinC3_updating")


def figS_attribution(out):
    fig, axes = plt.subplots(1, 2, figsize=(6.9, 3.0))
    for k, task in enumerate(("A1", "A2")):
        t = _t(out, f"attribution_{task}.csv").head(12).iloc[::-1]
        ax = axes[k]
        yy = np.arange(len(t))
        ax.plot(t.deployed_on_dev, yy, "o", color="#111111", ms=3.5, label="Deployed model, development data")
        ax.plot(t.deployed_on_C3, yy, "s", mfc="white", mec="#111111", ms=3.5, label="Deployed model, C3 data")
        pos = t.refit_on_C3 > 0
        ax.plot(t.refit_on_C3[pos], yy[pos.values], "^", color="#888888", ms=3.2, label="Refitted on C3")
        if (~pos).any():   # 2026-10-15: a zero cannot be drawn on a log axis; show it at the axis floor
            floor = min(t[["deployed_on_dev", "deployed_on_C3"]].min().min(), t.refit_on_C3[pos].min()) / 1.6
            ax.plot(np.full((~pos).sum(), floor), yy[(~pos).values], "<", mfc="white", mec="#888888", ms=3.4,
                    label="Refitted on C3: zero (drawn at axis floor)")
        ax.set_yticks(yy)
        ax.set_yticklabels([_lab(VAR_LABELS.get(v, v)) for v in t.variable], fontsize=6.0)
        ax.set_xscale("log")
        ax.set_xlabel("Mean |attribution| (log-odds, log scale)")
        ax.set_title(f"{TASK_TITLE[task]} ({SHORT.get(t.model.iloc[0], t.model.iloc[0])})", loc="left")
    fig.tight_layout()
    # shared legend above both panels; an inside legend covered the lowest-ranked variables of the retention panel
    h0, l0 = axes[0].get_legend_handles_labels()
    h1, l1 = axes[1].get_legend_handles_labels()
    extra = [(h, l) for h, l in zip(h1, l1) if l not in l0]   # the zero marker may exist in one panel only
    _legend_outside(fig, axes, h0 + [h for h, _ in extra], l0 + [l for _, l in extra], ncol=4 if extra else 3,
                    side="above")
    _save(fig, out, "figS_attribution")


def make_all(out):
    fig1_framework(out)
    fig2_performance(out)
    fig3_shift(out)
    fig4_updating(out)
    figS_fairness(out)
    figS_dca(out)
    figS_withinc3(out)
    figS_attribution(out)


In [ ]:
%%writefile {SRC_DIR}/rtw_tables.py
"""Format aggregated CSV outputs as Markdown tables (revision v2) and derive key numbers quoted in the text.

Display-layer consistency corrections 2026-10-15 (the CSV files and every value are unchanged): minus signs are U+2212
as in the text; model, design, decomposition-component and cohort codes are printed with the names used in the
manuscript; count columns get thousands separators; code-style column names are replaced by readable headers;
ranges and inequalities in labels use en dashes and typographic signs; "|" in cells is escaped instead of being
replaced by "/"; the C1 commuting-injury cell of Table S3 is "n/a" (a structural zero: C1 records only work accident
versus occupational disease); the inverse-probability-of-response-weighting sensitivity row is labelled IPRW.

Second display pass 2026-10-15 (consistency audit, fix round 2; the CSV files and every value are unchanged): readable
headers also for the remaining code-style columns of the tuning (Table S2), attrition (Table S3), sample-size (Table
S4), subgroup-audit (Table S11) and importance-weighting (Table S16) tables, and the domain-classifier column of the
DISDE information and importance-weighting tables (Tables S14 and S16) is headed "Domain classifier" with the values
"Logistic regression" and "LightGBM" instead of the codes LR and LGBM.

Third display pass 2026-10-15 (consistency audit close-out; the CSV files and every value are unchanged): every
remaining code-style or lowercase column name of the supplementary tables gets a readable header (Tables S1, S2, S7,
S9, S10, S12-S23), so no table mixes readable and code-style headers; "lo"/"hi" are printed as "Lower"/"Upper"
(also in the TPR/FPR gap columns of Table S11), single lowercase words are capitalised ("task" -> "Task"), and the
log-loss column of the full performance tables (Tables S5-S7) is headed "Log-loss".
"""
from __future__ import annotations

import os
import re
import sys

import numpy as np
import pandas as pd

from rtw_config import VAR_LABELS

SHORT = {"Null": "Null", "Status": "Status-only LR", "LR-small": "Parsimonious LR", "LR": "LR (scores)",
         "RidgeOH": "Ridge-OH", "RF": "Random forest", "XGB": "XGBoost", "LGBM": "LightGBM", "CatBoost": "CatBoost",
         "EBM": "EBM"}
ORDER = ["Null", "Status", "LR-small", "LR", "RidgeOH", "RF", "XGB", "LGBM", "CatBoost", "EBM"]
DESIGN = {"primary": "C1+C2 \u2192 C3 (primary)", "c1to3": "C1 \u2192 C3", "c2to3": "C2 \u2192 C3",
          "chain12": "C1 \u2192 C2 (chronological)", "loco1": "C2+C3 \u2192 C1 (leave one cohort out)",
          "loco2": "C1+C3 \u2192 C2 (leave one cohort out)"}
COMP = {"X_shift_source_side": "X-shift (source)", "YgivenX_shift": "Y|X shift", "X_shift_target_side": "X-shift (target)",
        "total": "Total change"}
LABELS = {"Age >=60": "Age \u226560", "Disability grade 1-7": "Disability grade 1\u20137",
          "Work ability, mean (0-10)": "Work ability, mean (0\u201310)", "<=39": "\u226439", ">=60": "\u226560",
          "40-49": "40\u201349", "50-59": "50\u201359", "Mild (8-14)": "Mild (8\u201314)",
          "Severe (1-7)": "Severe (1\u20137)", "Work ability (0-10)": "Work ability (0\u201310)"}
COUNT_COLS = ["n", "n_dev", "n_test", "events", "events_min_class", "test_events_min", "wave1_n", "wave2_n",
              "employed_w2", "n_strata", "single_unit_strata", "n_candidates", "reps", "Slope_n_undefined", "n_required",
              "n_crit1_shrinkage0.9", "n_crit2_optimism", "n_crit3_intercept", "n_source", "n_target", "parameters"]
ONE_DECIMAL = ["tune_seconds", "missing_C1_%", "missing_C2_%", "missing_C3_%"]
COLS = {"kish_ess": "Kish ESS", "ess_pct": "ESS (%)", "max_min_weight_ratio": "Max/min weight ratio", "n_strata": "Strata",
        "single_unit_strata": "Single-unit strata", "R2cs_anticipated": "Anticipated R\u00b2CS",
        "max_R2cs": "Max R\u00b2CS", "OE": "O/E", "coef_y": "Outcome coefficient", "lrt": "LRT statistic",
        "max_abs_smd_before": "Max \\|SMD\\| before", "max_abs_smd_after": "Max \\|SMD\\| after",  # headers: escaped
        "cv_logloss": "CV log-loss (tuning folds)", "repeated_cv_logloss": "Repeated-CV log-loss",
        "tune_seconds": "Tuning time (s)", "n_candidates": "Candidates"}
COLS.update({  # second display pass (2026-10-15, fix round 2): remaining code-style headers
    # Table S2 (tuning)
    "logloss_repeat1_tuning_folds": "Log-loss, repeat 1 (tuning folds)",
    "logloss_repeat2_independent_folds": "Log-loss, repeat 2 (independent folds)",
    "selected_v1_single_cv": "Chosen by single tuning CV (v1)",
    "selected_v2_repeated_cv": "Chosen by repeated CV (v2)",
    "selected_independent_repeat": "Chosen by independent repeat",
    # Table S3 (attrition)
    "wave1_n": "Wave-1 n", "wave2_n": "Wave-2 respondents", "response_pct": "Response (%)",
    "response_pct_employed_w1": "Response, employed at wave 1 (%)",
    "response_pct_not_employed_w1": "Response, not employed at wave 1 (%)",
    "missing_wt_w2_among_respondents": "Respondents without wave-2 weight",
    "missing_outcome_among_respondents": "Respondents without outcome",
    # Table S4 (sample size; Riley et al. criteria)
    "n_dev": "Development n", "events_min_class": "Development events (minority class)", "parameters": "Candidate parameters",
    "n_crit1_shrinkage0.9": "Required n, criterion 1 (shrinkage 0.9)",
    "n_crit2_optimism": "Required n, criterion 2 (optimism)",
    "n_crit3_intercept": "Required n, criterion 3 (intercept)", "n_required": "Required n (largest)",
    "expected_shrinkage_at_n": "Expected shrinkage at development n", "lr_oof_slope": "LR (scores) OOF calibration slope",
    "n_test": "C3 n", "test_events_min": "C3 events (minority class)",
    "OE_95CI_lower_factor": "O/E 95% CI lower factor", "OE_95CI_upper_factor": "O/E 95% CI upper factor",
    # Table S11 (subgroup audit)
    "TPR_implied": "Calibration-implied TPR",
    "gap_TPR": "TPR gap", "gap_TPR_lo": "TPR gap lo", "gap_TPR_hi": "TPR gap hi", "gap_TPR_q": "TPR gap q (BH)",
    "gap_FPR": "FPR gap", "gap_FPR_lo": "FPR gap lo", "gap_FPR_hi": "FPR gap hi", "gap_FPR_q": "FPR gap q (BH)",
    "gap_AUROC": "AUROC gap", "gap_AUROC_q": "AUROC gap q (BH)",
    # Tables S14 and S16 (domain classifier, balance)
    "classifier": "Domain classifier",
    "n_smd_gt_0.1_before": "Predictors with \\|SMD\\| > 0.1 before",   # headers: escaped pipes
    "n_smd_gt_0.1_after": "Predictors with \\|SMD\\| > 0.1 after"})
COLS.update({  # third display pass (2026-10-15, close-out): all remaining code-style or lowercase headers
    "lo": "Lower", "hi": "Upper", "reps": "Repetitions", "params": "Selected hyperparameters",
    "predictors": "Predictor set", "observed": "Observed employment (proportion)",
    # Tables S1, S7, S12 (predictors, within strata, prevalence, cohort effects)
    "missing_C1_%": "Missing, C1 (%)", "missing_C2_%": "Missing, C2 (%)", "missing_C3_%": "Missing, C3 (%)",
    "employed_w2": "Employed at wave 2", "prev": "Prevalence", "prev_weighted": "Prevalence (survey-weighted)",
    "log_odds": "Log odds ratio",
    # Table S9 (calibration change) and Table S10 (decision curves)
    "dev_oof": "Development (OOF)", "nb_model": "Net benefit (model)",
    "delta_vs_best_default": "\u0394 net benefit vs best default",
    # Table S14 (DISDE information)
    "domain_auroc": "Domain AUROC", "ess_source_shared": "ESS, source (shared-support weights)",
    "ess_target_shared": "ESS, target (shared-support weights)", "n_source": "Source n", "n_target": "Target n",
    "B": "Bootstrap resamples",
    # Table S15 (case-mix expected AUROC)
    "source_observed_auroc": "Observed AUROC, source (OOF)", "source_expected_auroc": "Case-mix expected AUROC, source",
    "target_observed_auroc": "Observed AUROC, target", "target_expected_auroc": "Case-mix expected AUROC, target",
    # Table S16 (full importance weighting; primary design, development = C1+C2, target = C3)
    "cap_pct": "Weight cap (percentile)", "max_w": "Max weight", "prev_dev": "Prevalence, development",
    "prev_target": "Prevalence, C3",
    "weighted_CITL": "IW CITL, development (OOF)", "weighted_Slope": "IW slope, development (OOF)",
    "weighted_AUROC": "IW AUROC, development (OOF)",
    "target_CITL": "CITL in C3", "target_Slope": "Slope in C3", "target_AUROC": "AUROC in C3",
    # Table S17 (label-free recalibration): change from no update (|CITL| for CITL, |slope - 1| for the slope)
    "improvement_vs_none": "Change from no update", "imp_lo": "Change lower", "imp_hi": "Change upper",
    # Tables S18-S19 (lagged and within-C3 updating; medians and percentiles over repetitions)
    "AUROC_median": "AUROC (median)", "Brier_median": "Brier (median)", "CITL_median": "CITL (median)",
    "Slope_median": "Slope (median)", "ICI_median": "ICI (median)",
    "ICI_p2_5": "ICI (2.5th percentile)", "ICI_p97_5": "ICI (97.5th percentile)",
    "share_better_ICI": "Share improved, ICI", "share_better_Brier": "Share improved, Brier",
    "share_better_absCITL": "Share improved, \\|CITL\\|",               # headers: escaped pipes
    "share_better_slope": "Share improved, \\|slope \u2212 1\\|",
    "mean_absCITL": "Mean \\|CITL\\|", "Slope_n_undefined": "Slope undefined (repetitions)",
    "closed_choice_none": "Closed testing chose no update (share)",
    "closed_choice_intercept": "Closed testing chose intercept (share)",
    "closed_choice_logistic": "Closed testing chose logistic (share)",
    # Table S22 (seed variability)
    "MeanPred": "Mean prediction", "NoSkillAUPRC": "No-skill AUPRC",
    # Table S23 (attribution: mean |SHAP| importance)
    "deployed_on_dev": "Importance, deployed model on development",
    "deployed_on_C3": "Importance, deployed model on C3",
    "diff_C3_minus_dev": "Difference (C3 \u2212 development)", "diff_lo": "Difference lower",
    "diff_hi": "Difference upper", "refit_on_C3": "Importance, model refitted on C3",
    "top10_overlap_refit": "Top-10 overlap, development vs C3 refit", "split_R": "Split-half repetitions"})
COLS.update({f"gap_{g}_{k}": f"{g} gap {v}" for g in ("TPR", "FPR") for k, v in (("lo", "lower"), ("hi", "upper"))})
IW_COLS = {"prev_weighted": "IW prevalence, development"}   # Table S16 only (Table S12: survey-weighted prevalence)
STAB = {"deployed_dev_vs_C3": "deployed model, development vs C3", "dev_vs_C3_refit": "development vs C3 refit",
        "within_C3": "within C3", "within_dev": "within development",
        "between": "between development and C3"}               # Table S23 (split-half rank correlations)
STAB_STAT = {"_median": ", median", "_p2_5": ", 2.5th percentile", "_p97_5": ", 97.5th percentile"}
DOMAIN_CLF = {"LR": "Logistic regression", "LGBM": "LightGBM"}   # domain-classifier codes (rtw_shift)
TASKS = ("A1", "A2", "P")


def T(out, name):
    return pd.read_csv(os.path.join(out, "tables", name))


def f(x, d=3):
    if pd.isna(x):
        return "–"
    s = f"{x:.{d}f}"
    while float(s) == 0 and x != 0 and d < 6:
        d += 1
        s = f"{x:.{d}f}"
    if s.startswith("-") and float(s) == 0:
        return s[1:]
    return "\u2212" + s[1:] if s.startswith("-") else s   # U+2212 minus, as in the text


def ci(e, lo, hi, d=3):
    if pd.isna(e):
        return "–"
    if pd.isna(lo):
        return f(e, d)
    return f"{f(e, d)} ({f(lo, d)}, {f(hi, d)})"


def md(df):
    cols = list(df.columns)
    out = ["| " + " | ".join(cols) + " |", "|" + "|".join(["---"] * len(cols)) + "|"]
    for _, r in df.iterrows():
        out.append("| " + " | ".join(str(r[c]).replace("|", "\\|") for c in cols) + " |")
    return "\n".join(out)


def primary(out, task):
    t = T(out, "tuning_primary.csv")
    t = t[t.task == task]
    return t.loc[t.repeated_cv_logloss.idxmin(), "model"]


def table1(out):
    t = T(out, "table1_characteristics.csv")
    rows = []
    for _, r in t.iterrows():
        row = {"Characteristic": LABELS.get(r.characteristic, r.characteristic)}
        for c in (1, 2, 3):
            u, w = r[f"C{c}_unw"], r.get(f"C{c}_w", np.nan)
            if c == 1 and r.characteristic == "Commuting injury":
                row["C1"] = "n/a"   # structural zero: C1 records only work accident vs occupational disease
            elif r.kind == "n":
                row[f"C{c}"] = f"{int(u):,}"
            elif r.kind == "pct_only":
                row[f"C{c}"] = f"{u:.1f}"
            elif r.kind == "mean":
                row[f"C{c}"] = f"{u:.2f} ({w:.2f})"
            else:
                row[f"C{c}"] = f"{u:.1f} ({w:.1f})"
        rows.append(row)
    return md(pd.DataFrame(rows).rename(columns={"C1": "Cohort 1", "C2": "Cohort 2", "C3": "Cohort 3"}))


def table2(out, models_per_task=None):
    rows = []
    for task in TASKS:
        u = T(out, f"perf_C3_{task}_unweighted.csv")
        w = T(out, f"perf_C3_{task}_weighted.csv")
        prim = primary(out, task)
        base = ["Status"] if task == "P" else ["Null"]          # round 2: trimmed (full tables in S5-S7)
        models = list(dict.fromkeys(base + ["RidgeOH", prim]))
        for m in models:
            gu = u[u.model == m].set_index("metric")
            gw = w[w.model == m].set_index("metric")
            if len(gu) == 0:
                continue
            nul = m == "Null"
            rows.append({"Task": task, "Model": SHORT[m] + (" *" if m == prim else ""),
                         "AUROC (sample)": "–" if nul else ci(*gu.loc["AUROC", ["estimate", "lo", "hi"]]),
                         "AUROC (population)": "–" if nul else ci(*gw.loc["AUROC", ["estimate", "lo", "hi"]]),
                         "CITL (sample)": ci(*gu.loc["CITL", ["estimate", "lo", "hi"]], d=2),
                         "Slope (sample)": "–" if nul else ci(*gu.loc["Slope", ["estimate", "lo", "hi"]], d=2),
                         "CITL (population)": ci(*gw.loc["CITL", ["estimate", "lo", "hi"]], d=2),
                         "ICI (sample)": "–" if nul else f(gu.loc["ICI", "estimate"])})
    return md(pd.DataFrame(rows))


def table3(out):
    dis = T(out, "shift_disde.csv")
    iw = T(out, "update_iw_recalibration.csv")
    rows = []
    for task in ("A1", "A2", "P"):
        prim = primary(out, task)
        for des in ("C1 -> C3", "C2 -> C3", "C1+C2 -> C3"):
            pv = dis[(dis.task == task) & (dis.design == des) & (dis.metric == "Prevalence")].set_index("component")
            m = prim if ((dis.task == task) & (dis.design == des) & (dis.model == prim)).any() else "XGB"
            cl = dis[(dis.task == task) & (dis.design == des) & (dis.metric == "CITL") & (dis.model == m)].set_index("component")
            ir = iw[(iw.task == task) & (iw.design == des) & (iw.model == m)]
            def g(meth, k):
                r = ir[(ir.method == meth) & (ir.metric == k)]
                return r.iloc[0].estimate if len(r) else np.nan
            if len(pv) == 0:
                continue
            rows.append({"Task": task, "Design": des.replace(" -> ", " → "),
                         "Δ prevalence": ci(*pv.loc["total", ["estimate", "lo", "hi"]]),
                         "yx_prev": ci(*pv.loc["YgivenX_shift", ["estimate", "lo", "hi"]]),
                         "CITL in C3": f(cl.loc["total", "target_value"], 2),
                         "yx_citl": ci(*cl.loc["YgivenX_shift", ["estimate", "lo", "hi"]], d=2)})
    df = pd.DataFrame(rows)   # round 2: IW columns moved to Table S17 / Fig. 4a; prevalence rows are model-free
    df.columns = ["Task", "Design", "Δ employment prevalence (model-free)", "Y\\|X component (prev.)",
                  f"CITL in C3 ({SHORT[primary(out, 'A2')]})", "Y\\|X component (CITL)"]   # escaped pipe for Markdown
    return md(df)


def perf_full(out, task, lab):
    u = T(out, f"perf_C3_{task}_{lab}.csv")
    dev = T(out, f"perf_devOOF_{task}.csv").set_index("model")
    rows = []
    for m in ORDER:
        g = u[u.model == m].set_index("metric")
        if len(g) == 0:
            continue
        rows.append({"Model": SHORT[m], "Dev AUROC (OOF)": f(dev.loc[m, "AUROC"]) if m in dev.index else "–",
                     **{("Log-loss" if k == "LogLoss" else k):   # readable header (2026-10-15, close-out)
                        ci(*g.loc[k, ["estimate", "lo", "hi"]], d=4 if k in ("Brier", "LogLoss") else 3)
                        for k in ("AUROC", "AUPRC", "Brier", "LogLoss", "CITL", "Slope", "ICI", "E50", "E90", "ECE")},
                     "No-skill AUPRC": f(g.loc["NoSkillAUPRC", "estimate"]),
                     "Slope undefined (boot)": int(g.loc["Slope", "n_undefined"])})
    return md(pd.DataFrame(rows))


def diffs(out, task, lab):
    d = T(out, f"perf_C3_diffs_{task}_{lab}.csv")
    d = d[d.metric.isin(["AUROC", "Brier", "ICI"])]
    rows = [{"Model": SHORT[r.model], "Reference": SHORT[r.reference], "Metric": r.metric,
             "Difference": ci(r.estimate, r.lo, r.hi, 4)} for _, r in d.iterrows()]
    return md(pd.DataFrame(rows))


def _header(c):
    """Readable column header (display only; the CSV column names are unchanged)."""
    if c in COLS:
        return COLS[c]
    for pre, sym in (("rho_", "\u03c1 "), ("wtau_", "\u03c4w ")):
        if c.startswith(pre):
            rest, stat = c[len(pre):], ""
            for suf, lab in STAB_STAT.items():
                if rest.endswith(suf):
                    rest, stat = rest[: -len(suf)], lab
            return sym + STAB.get(rest, rest) + stat
    if re.fullmatch(r"[a-z]+", c) and c not in ("n", "p"):
        return c.capitalize()
    return c


def _count(x):
    return "\u2013" if pd.isna(x) else f"{int(round(float(x))):,}"


def generic(out, name, cols=None, d=3, rename=None):
    t = T(out, name)
    if cols:
        t = t[cols]
    t = t.copy()
    # display names (2026-10-15); the CSV values used by the manuscript placeholders are unchanged
    for c in ("model", "reference"):
        if c in t.columns:
            t[c] = t[c].map(lambda x: SHORT.get(x, x))
    if "classifier" in t.columns:   # domain classifier (not a prediction model): spelled out (2026-10-15)
        t["classifier"] = t["classifier"].map(lambda x: DOMAIN_CLF.get(x, x))
    if "design" in t.columns:
        t["design"] = t["design"].map(lambda x: DESIGN.get(x, x))
    for c in ("design", "analysis"):
        if c in t.columns:
            t[c] = t[c].astype(str).str.replace(" -> ", " \u2192 ", regex=False)
    if "component" in t.columns:
        t["component"] = t["component"].map(lambda x: COMP.get(x, x))
    if "cohort" in t.columns and pd.api.types.is_integer_dtype(t["cohort"]):
        t["cohort"] = t["cohort"].map(lambda x: f"C{x}")
    for c in ("level", "reference", "label", "characteristic"):
        if c in t.columns:
            t[c] = t[c].map(lambda x: LABELS.get(x, x))
    for c in COUNT_COLS:
        if c in t.columns and pd.api.types.is_numeric_dtype(t[c]):
            t[c] = t[c].map(_count)
    for c in ONE_DECIMAL:
        if c in t.columns and pd.api.types.is_numeric_dtype(t[c]):
            t[c] = t[c].map(lambda x: f(x, 1))
    for c in t.columns:
        if pd.api.types.is_float_dtype(t[c]):
            t[c] = t[c].map(lambda x: f(x, d))
    return md(t.rename(columns={c: (rename or {}).get(c, _header(c)) for c in t.columns}))


def key_numbers(out):
    rows = []
    t1 = T(out, "table1_characteristics.csv").set_index("characteristic")
    for c in (1, 2, 3):
        rows.append((f"response_pct_C{c}", t1.loc["Wave-2 response, %", f"C{c}_unw"], "table1_characteristics.csv"))
    for task in TASKS:
        prim = primary(out, task)
        rows.append((f"primary_is_{task}_{prim}", 1, "tuning_primary.csv"))
        for lab in ("unweighted", "weighted"):
            dd = T(out, f"dca_delta_{task}_{lab}.csv")
            for _, r in dd.iterrows():
                rows.append((f"dca_{task}_{lab}_{r.model}_pt{r.threshold:.2f}", r.delta_vs_best_default, f"dca_delta_{task}_{lab}.csv"))
    k = pd.DataFrame(rows, columns=["name", "value", "source"])
    k.to_csv(os.path.join(out, "tables", "key_numbers.csv"), index=False)
    return k


def make_all(out):
    key_numbers(out)
    parts = {"Main Table 1": table1(out), "Main Table 2": table2(out), "Main Table 3": table3(out)}
    for task in TASKS:
        for lab in ("unweighted", "weighted"):
            parts[f"Performance {task} {lab}"] = perf_full(out, task, lab)
            parts[f"Differences {task} {lab}"] = diffs(out, task, lab)
    parts["Tuning primary"] = generic(out, "tuning_primary.csv", d=4)
    parts["Tuning secondary"] = generic(out, "tuning_secondary.csv", d=4)
    parts["Sample size"] = generic(out, "sample_size_riley.csv")
    parts["Calibration change"] = "\n\n".join(generic(out, f"calibration_change_{t}.csv") for t in TASKS)
    parts["DCA delta"] = "\n\n".join(generic(out, f"dca_delta_{t}_{lab}.csv", d=4) for t in TASKS
                                     for lab in ("unweighted", "weighted"))
    parts["Prevalence"] = generic(out, "shift_prevalence.csv")
    parts["Cohort effects"] = generic(out, "shift_cohort_effects.csv")
    parts["Outcome in domain"] = generic(out, "shift_outcome_in_domain_test.csv")
    parts["Adversarial"] = generic(out, "shift_adversarial.csv")
    parts["DISDE"] = generic(out, "shift_disde.csv", ["task", "design", "model", "metric", "component", "estimate", "lo", "hi"], d=4)
    parts["DISDE info"] = generic(out, "shift_disde_info.csv")
    parts["Case mix"] = generic(out, "shift_casemix.csv")
    parts["IW sensitivity"] = generic(out, "shift_iw_sensitivity.csv", rename=IW_COLS)
    parts["IW recalibration"] = generic(out, "update_iw_recalibration.csv",
                                        ["task", "design", "model", "method", "metric", "estimate", "lo", "hi",
                                         "improvement_vs_none", "imp_lo", "imp_hi"], d=4)
    lg = T(out, "update_lagged_summary.csv")
    parts["Lagged updating"] = generic(out, "update_lagged_summary.csv",
                                       ["task", "model", "method", "n", "reps", "AUROC_median", "Brier_median", "CITL_median",
                                        "Slope_median", "ICI_median", "ICI_p2_5", "ICI_p97_5", "share_better_ICI",
                                        "share_better_Brier", "Slope_n_undefined"] +
                                       [c for c in lg.columns if c.startswith("closed_choice")], d=4)
    wc = T(out, "update_withinC3_summary.csv")
    parts["Within-C3 updating"] = generic(out, "update_withinC3_summary.csv",
                                          ["task", "model", "method", "n", "reps", "AUROC_median", "Brier_median",
                                           "CITL_median", "Slope_median", "ICI_median", "mean_absCITL",
                                           "share_better_absCITL", "share_better_slope", "share_better_Brier",
                                           "Slope_n_undefined"] + [c for c in wc.columns if c.startswith("closed_choice")], d=4)
    fa = T(out, "fairness.csv")
    keep = ["task", "model", "weighting", "threshold", "group", "level", "n", "observed", "OE", "AUROC", "CITL", "TPR",
            "TPR_implied", "FPR", "PPV", "NPV", "gap_TPR", "gap_TPR_lo", "gap_TPR_hi", "gap_TPR_q", "gap_FPR",
            "gap_FPR_lo", "gap_FPR_hi", "gap_FPR_q", "gap_AUROC", "gap_AUROC_q"]
    parts["Fairness"] = generic(out, "fairness.csv", [c for c in keep if c in fa.columns])
    parts["Attribution"] = "\n\n".join(generic(out, f"attribution_{t}.csv") for t in TASKS)
    parts["Attribution stability"] = "\n\n".join(generic(out, f"attribution_stability_{t}.csv") for t in TASKS)
    parts["Sensitivity"] = "\n\n".join(generic(out, f"sensitivity_{t}.csv", ["analysis", "model", "metric", "estimate", "lo", "hi", "n"])
                                       for t in TASKS).replace("| IPCW training and evaluation |",
                                                               "| IPRW, training and evaluation |")
    sd = [n for n in os.listdir(os.path.join(out, "tables")) if n.startswith("seed_variability_")]
    parts["Seed variability"] = "\n\n".join(generic(out, n) for n in sorted(sd))
    parts["Secondary designs"] = generic(out, "secondary_designs.csv",
                                         ["task", "analysis", "model", "metric", "estimate", "lo", "hi", "n", "events"])
    parts["Predictors"] = generic(out, "tableS1_predictors.csv")
    parts["Within strata"] = generic(out, "perf_C3_P_within_strata.csv",
                                     ["stratum", "weighting", "model", "metric", "estimate", "lo", "hi", "n", "employed_w2"])
    parts["Attrition"] = generic(out, "attrition.csv", d=1)
    parts["Weights design"] = generic(out, "weights_design.csv", d=1)
    parts["DISDE restricted"] = generic(out, "shift_disde_restricted.csv",
                                        ["task", "design", "predictors", "model", "metric", "component", "estimate",
                                         "lo", "hi"], d=4)
    with open(os.path.join(out, "tables_formatted.md"), "w", encoding="utf-8") as fh:
        for k, v in parts.items():
            fh.write(f"## {k}\n\n{v}\n\n")
    return parts


if __name__ == "__main__":
    make_all(sys.argv[1])


In [ ]:
%%writefile {SRC_DIR}/run_all.py
"""Run the complete analysis pipeline (revision v2).

Environment variables:
    DATA_ROOT       folder containing the PSWCI cohort folders (raw SPSS files)
    RTW_OUT_DIR     where aggregated tables/figures are written
    RTW_CACHE_DIR   cache for fitted models / row-level predictions (never share)
    RTW_USE_PARQUET 1 = read DATA_ROOT/00_derived_data/pswci_harmonized_long.parquet if present
    RTW_STAGES      comma-separated subset of stages (default: all)
"""
from __future__ import annotations

import os
import sys
import time

os.environ.setdefault("PYTHONWARNINGS", "ignore")
import warnings  # noqa: E402

warnings.filterwarnings("ignore")

import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))

from rtw_config import B_BOOT, B_SHIFT, N_JOBS, R_UPDATE, SEED, paths  # noqa: E402
from rtw_data import build_analytic, load_long, task_frame  # noqa: E402
import rtw_analyses as A  # noqa: E402
import rtw_shift as SH  # noqa: E402
import rtw_update as U  # noqa: E402
from rtw_develop import designs_for, develop, params_table  # noqa: E402
from rtw_fair import fairness  # noqa: E402
from rtw_models import ONEHOT_MODELS  # noqa: E402

TASKS = ("A1", "A2", "P")
SHIFT_MODELS = ["LR", "RidgeOH", "XGB", "LGBM", "CatBoost", "EBM"]
SECONDARY = [("c1to3", (1,), 3, SHIFT_MODELS), ("c2to3", (2,), 3, SHIFT_MODELS),
             ("chain12", (1,), 2, ["LR", "RidgeOH", "XGB"]), ("loco1", (2, 3), 1, ["LR", "RidgeOH", "XGB"]),
             ("loco2", (1, 3), 2, ["LR", "RidgeOH", "XGB"])]
DESIGN_LABEL = {"primary": "C1+C2 -> C3", "c1to3": "C1 -> C3", "c2to3": "C2 -> C3", "chain12": "C1 -> C2",
                "loco1": "C2+C3 -> C1", "loco2": "C1+C3 -> C2"}


def main():
    np.random.seed(SEED)
    data_root, out, cache = paths()
    stages = os.environ.get("RTW_STAGES", "all").split(",")
    run = lambda s: "all" in stages or s in stages  # noqa: E731
    logf = open(os.path.join(out, "run_log.txt"), "a", encoding="utf-8")

    def log(msg):
        line = f"{time.strftime('%Y-%m-%d %H:%M:%S')} {msg}"
        print(line, flush=True)
        logf.write(line + "\n")
        logf.flush()

    t0 = time.time()
    log(f"=== v2 run start; stages={stages}; B_BOOT={B_BOOT} B_SHIFT={B_SHIFT} R_UPDATE={R_UPDATE} N_JOBS={N_JOBS}")
    a = build_analytic(load_long(data_root))
    T = lambda n: os.path.join(out, "tables", n)  # noqa: E731
    if run("describe"):
        A.descriptives(a, out)
        A.predictor_table(a, out)
        SH.prevalence_table(a).to_csv(T("shift_prevalence.csv"), index=False)
    res = {t: develop(a, t, (1, 2), 3, tag="primary", cache_dir=cache, log=log) for t in TASKS}
    pd.concat([params_table(r) for r in res.values()]).to_csv(T("tuning_primary.csv"), index=False)
    log(f"--- primary development done ({time.time() - t0:.0f}s)")
    if run("evaluate"):
        A.riley(res, a, out)
        for t in TASKS:
            A.evaluate_task(res[t], a, out, log=log)
        log(f"--- evaluation done ({time.time() - t0:.0f}s)")
    sec = {}
    for tag, trc, tec, models in SECONDARY:
        for t in TASKS:
            sec[(tag, t)] = develop(a, t, trc, tec, reduced=True, tag=tag, cache_dir=cache, log=log, models=models,
                                    select_repeats=1)
    log(f"--- secondary development done ({time.time() - t0:.0f}s)")
    if run("secondary"):
        rows = []
        for (tag, t), r in sec.items():
            rows.append(A.quick_eval(r["y_test"], A._ordered(r), t, B_BOOT, DESIGN_LABEL[tag],
                                     extra={"design": tag, "primary_in_design": r["primary"]}))
        for t in TASKS:
            rows.append(A.quick_eval(res[t]["y_test"], A._ordered(res[t]), t, B_BOOT, DESIGN_LABEL["primary"],
                                     extra={"design": "primary", "primary_in_design": res[t]["primary"]}))
        pd.concat(rows, ignore_index=True).to_csv(T("secondary_designs.csv"), index=False)
        pd.concat([params_table(r, repeats=1) for r in sec.values()]).to_csv(T("tuning_secondary.csv"), index=False)
        log(f"--- secondary evaluation done ({time.time() - t0:.0f}s)")
    if run("shift"):
        ce = pd.concat([SH.cohort_effects(a, t) for t in TASKS])
        ce.to_csv(T("shift_cohort_effects.csv"), index=False)
        it = pd.DataFrame([SH.outcome_in_domain_test(a, t, s) for t in TASKS for s in ((1,), (2,), (1, 2))])
        it.to_csv(T("shift_outcome_in_domain_test.csv"), index=False)
        pairs = [((1,), (3,)), ((2,), (3,)), ((1, 2), (3,)), ((1,), (2,))]
        adv, drv = SH.adversarial(a, pairs, B=B_BOOT)
        adv2, drv2 = SH.adversarial(a, pairs, B=B_BOOT, restricted=True)
        pd.concat([adv, adv2]).to_csv(T("shift_adversarial.csv"), index=False)
        pd.concat([drv, drv2]).to_csv(T("shift_adversarial_drivers.csv"), index=False)
        dis, info, cm, iws = [], [], [], []
        for t in TASKS:
            d = task_frame(a, t)
            D = designs_for(d, t)
            for tag, r in (("primary", res[t]), ("c1to3", sec[("c1to3", t)]), ("c2to3", sec[("c2to3", t)])):
                Xn = D["num"][0]
                Xs, Xt = Xn.loc[r["dev_index"]], Xn.loc[r["test_index"]]
                models = [m for m in ("RidgeOH", "XGB", res[t]["primary"]) if m in r["oof"]]
                models = list(dict.fromkeys(models))
                tab, inf = SH.disde(Xs, r["y_dev"], {m: r["oof"][m] for m in models}, Xt, r["y_test"],
                                    {m: r["test"][m] for m in models})
                tab["task"], tab["design"] = t, DESIGN_LABEL[tag]
                dis.append(tab)
                info.append({**inf, "task": t, "design": DESIGN_LABEL[tag]})
                for m in models:
                    cm.append({"task": t, "design": DESIGN_LABEL[tag], "model": m,
                               "source_observed_auroc": A.all_metrics(r["y_dev"], r["oof"][m], t, which=("AUROC",))["AUROC"],
                               "source_expected_auroc": SH.casemix_auroc(r["oof"][m]),
                               "target_observed_auroc": A.all_metrics(r["y_test"], r["test"][m], t, which=("AUROC",))["AUROC"],
                               "target_expected_auroc": SH.casemix_auroc(r["test"][m])})
                if tag == "primary":
                    m = res[t]["primary"]
                    iw = SH.iw_sensitivity(Xs, r["y_dev"], r["oof"][m], Xt, r["y_test"], r["test"][m])
                    iw["task"], iw["model"] = t, m
                    iws.append(iw)
            log(f"[shift {t}] DISDE done ({time.time() - t0:.0f}s)")
        pd.concat(dis).to_csv(T("shift_disde.csv"), index=False)
        pd.DataFrame(info).to_csv(T("shift_disde_info.csv"), index=False)
        pd.DataFrame(cm).to_csv(T("shift_casemix.csv"), index=False)
        pd.concat(iws).to_csv(T("shift_iw_sensitivity.csv"), index=False)
        log(f"--- shift done ({time.time() - t0:.0f}s)")
    if run("iwrecal"):
        rows = []
        for t in TASKS:
            d = task_frame(a, t)
            Xn = designs_for(d, t)["num"][0]
            for tag, r in (("primary", res[t]), ("c1to3", sec[("c1to3", t)]), ("c2to3", sec[("c2to3", t)]),
                           ("chain12", sec[("chain12", t)])):
                models = list(dict.fromkeys([m for m in ("LR", "RidgeOH", "XGB", res[t]["primary"]) if m in r["oof"]]))
                tab = U.iw_recal_eval(Xn.loc[r["dev_index"]], r["y_dev"], {m: r["oof"][m] for m in models},
                                      Xn.loc[r["test_index"]], r["y_test"], {m: r["test"][m] for m in models})
                tab["task"], tab["design"] = t, DESIGN_LABEL[tag]
                rows.append(tab)
            log(f"[iw-recal {t}] done ({time.time() - t0:.0f}s)")
        pd.concat(rows).to_csv(T("update_iw_recalibration.csv"), index=False)
    if run("lagged"):
        longs, summ = [], []
        for t in ("A1", "A2", "P"):
            r1 = sec[("c1to3", t)]
            m = res[t]["primary"] if res[t]["primary"] in r1["estimators"] else "XGB"
            d = task_frame(a, t)
            D = designs_for(d, t)
            Xn, Xo = D["num"][0], D["oh"][0]
            i1 = d.index[d.cohort == 1]
            i2 = d.index[d.cohort == 2]
            i3 = d.index[d.cohort == 3]
            Xb = Xo if m in ONEHOT_MODELS else Xn
            lg = U.lagged_updating(m, r1["params"][m], Xb.loc[i1], d.loc[i1, "y"].values, Xb.loc[i2],
                                   d.loc[i2, "y"].values, Xb.loc[i3], d.loc[i3, "y"].values, Xo.loc[i2], Xo.loc[i3],
                                   r1["estimators"][m], R=R_UPDATE)
            lg["task"], lg["model"] = t, m
            longs.append(lg)
            s = U.summarise_updating(lg, "No update (C1 model)")
            s["task"], s["model"] = t, m
            summ.append(s)
            log(f"[lagged {t}] {m} done ({time.time() - t0:.0f}s)")
        pd.concat(longs).to_csv(T("update_lagged_reps.csv"), index=False)
        pd.concat(summ).to_csv(T("update_lagged_summary.csv"), index=False)
    if run("withinc3"):
        longs, summ = [], []
        for t in ("A1", "A2"):
            r = res[t]
            m = r["primary"]
            d = task_frame(a, t)
            D = designs_for(d, t)
            X = D["oh" if m in ONEHOT_MODELS else "num"][0]
            lg = U.within_c3_updating(m, r["params"][m], X.loc[r["dev_index"]], r["y_dev"], X.loc[r["test_index"]],
                                      r["y_test"], r["test"][m], D["oh"][0].loc[r["test_index"]], R=R_UPDATE,
                                      sizes=(100, 250, 500, 1000) if t == "A1" else (100, 250, 500))
            lg["task"], lg["model"] = t, m
            longs.append(lg)
            s = U.summarise_updating(lg, "No update")
            s["task"], s["model"] = t, m
            summ.append(s)
            log(f"[within-C3 {t}] done ({time.time() - t0:.0f}s)")
        pd.concat(longs).to_csv(T("update_withinC3_reps.csv"), index=False)
        pd.concat(summ).to_csv(T("update_withinC3_summary.csv"), index=False)
    if run("fairness"):
        from rtw_config import FAIR_THRESHOLD
        rows = []
        for t in TASKS:
            r = res[t]
            te = task_frame(a, t).loc[r["test_index"]]
            for m in dict.fromkeys([r["primary"], "RidgeOH"]):
                rows.append(fairness(te, r["y_test"], r["test"][m], t, m, FAIR_THRESHOLD[t], B=B_BOOT))
                rows.append(fairness(te, r["y_test"], r["test"][m], t, m, FAIR_THRESHOLD[t], B=B_BOOT,
                                     w=te["wt_w2"].values, strata=te["strata"].values, label="weighted"))
        pd.concat(rows).to_csv(T("fairness.csv"), index=False)
        log(f"--- fairness done ({time.time() - t0:.0f}s)")
    if run("attribution"):
        for t in ("A1", "A2", "P"):
            A.attribution_stability(res[t], a, out, log=log)
    if run("extras"):
        A.within_strata(res["P"], a, out)
        A.attrition(a, out)
        A.weights_design(a, out)                     # round 2: Kish ESS, single-unit strata
        A.disde_restricted(res, sec, a, out, log=log)  # round 2: restricted-set prevalence decomposition
        log(f"--- extras done ({time.time() - t0:.0f}s)")
    if run("sensitivity"):
        for t in ("A1", "A2", "P"):
            A.sensitivity(res[t], a, out, log=log)
            A.seed_variability(res[t], a, out)
        log(f"--- sensitivity done ({time.time() - t0:.0f}s)")
    if run("report"):
        import rtw_figures
        import rtw_tables
        rtw_figures.make_all(out)
        rtw_tables.make_all(out)
    log(f"=== v2 run complete in {time.time() - t0:.0f}s")
    logf.close()


if __name__ == "__main__":
    main()


In [ ]:
%%writefile {SRC_DIR}/export_models.py
"""Export the fitted development models (TRIPOD+AI item 22) without any person-level data.

For each task (A1, A2, P) and the primary development design (C1+C2):
  * CatBoost (primary model): native JSON model file (tree splits, borders, leaf values) plus the imputation medians.
    Training counts per leaf (`leaf_weights`), the training-loss history, file names, GUID and timestamp are removed
    (round 2: small-cell protection; leaves of the pooled-task depth-3 trees can hold < 5 training records). These
    fields are not needed for scoring; predictions are checked to be unchanged after stripping.
  * ridge-OH, LR-small and (pooled task) the status-only model: intercept and coefficients on the original scale with
    the imputation medians (scaling constants are not exported, so category counts cannot be derived).
Only fitted parameters and aggregate constants are written. Usage:
    python export_models.py CACHE_DIR OUT_DIR
"""
import json
import os
import sys

import joblib
import numpy as np
import pandas as pd


def feature_names(pipe, n_in):
    imp = pipe.named_steps["imp"]
    names_in = list(getattr(imp, "feature_names_in_", [f"x{i}" for i in range(n_in)]))
    try:
        out = list(imp.get_feature_names_out(names_in))
    except Exception:  # pragma: no cover
        out = names_in + [f"missingindicator_{names_in[i]}" for i in imp.indicator_.features_]
    return names_in, out


def export_linear(pipe, model, task, out):
    imp, sc, clf = pipe.named_steps["imp"], pipe.named_steps["sc"], pipe.named_steps["clf"]
    names_in, names_out = feature_names(pipe, len(imp.statistics_))
    beta = clf.coef_.ravel()
    mu, sd = sc.mean_, np.where(sc.scale_ > 0, sc.scale_, 1.0)
    b_orig = beta / sd
    a_orig = float(clf.intercept_[0] - np.sum(beta * mu / sd))
    med = dict(zip(names_in, imp.statistics_))
    rows = [{"feature": "(intercept)", "imputation_median": np.nan, "coefficient": a_orig}]
    for f, bo in zip(names_out, b_orig):
        rows.append({"feature": f, "imputation_median": med.get(f, np.nan), "coefficient": bo})
    pd.DataFrame(rows).to_csv(os.path.join(out, f"model_{task}_{model}_coefficients.csv"), index=False)


def check_catboost(clf, path, n_features, n=2000, seed=1):
    """Predictions of the stripped JSON model must equal those of the fitted model (random inputs)."""
    from catboost import CatBoostClassifier
    rng = np.random.default_rng(seed)
    X = rng.normal(0, 3, size=(n, n_features))
    m2 = CatBoostClassifier()
    m2.load_model(path, format="json")
    d = np.max(np.abs(clf.predict_proba(X)[:, 1] - m2.predict_proba(X)[:, 1]))
    if d > 1e-9:
        raise RuntimeError(f"stripped model differs ({d})")


def export_catboost(pipe, task, out):
    imp, clf = pipe.named_steps["imp"], pipe.named_steps["clf"]
    names_in, names_out = feature_names(pipe, len(imp.statistics_))
    path = os.path.join(out, f"model_{task}_CatBoost.json")
    clf.save_model(path, format="json")
    with open(path) as fh:
        js = json.load(fh)
    for tree in js["oblivious_trees"]:
        tree.pop("leaf_weights", None)
    for k in ("training", "train_finish_time", "model_guid", "output_options"):
        js["model_info"].pop(k, None)
    with open(path, "w") as fh:
        json.dump(js, fh)
    check_catboost(clf, path, len(names_out))
    pd.DataFrame({"feature": names_in, "imputation_median": imp.statistics_}).to_csv(
        os.path.join(out, f"model_{task}_CatBoost_imputation.csv"), index=False)
    with open(os.path.join(out, f"model_{task}_CatBoost_inputs.json"), "w") as f:
        json.dump({"input_order_after_imputer": names_out,
                   "note": "Predicted probability = employed at wave 2. Inputs: median-imputed predictors followed by "
                           "missing-value indicators, in this order (design matrix from rtw_data.design_matrix)."}, f,
                  indent=1)


def main(cache, out):
    os.makedirs(out, exist_ok=True)
    for task in ("A1", "A2", "P"):
        r = joblib.load(os.path.join(cache, f"dev_primary_{task}.joblib"))
        export_catboost(r["estimators"]["CatBoost"], task, out)
        for m in ("RidgeOH", "LR-small", "Status"):
            if m in r["estimators"]:
                export_linear(r["estimators"][m], m, task, out)
    print("models exported to", out)


if __name__ == "__main__":
    main(*sys.argv[1:3])


In [ ]:
# 4. Build the harmonized long file from the raw SPSS data and show aggregated checks
sys.path.insert(0, SRC_DIR)
import importlib, pandas as pd
import pswci_harmonize
long = pswci_harmonize.build_harmonized(DATA_ROOT)
print(long.shape)
print(long.groupby(["cohort", "wave"])["participated"].agg(["size", "sum"]))
pq = os.path.join(DATA_ROOT, "00_derived_data", "pswci_harmonized_long.parquet")
if os.path.exists(pq):   # optional consistency check against the shared pre-built file
    ref = pd.read_parquet(pq)
    common = [c for c in ref.columns if c in long.columns]
    same = ref[common].reset_index(drop=True).equals(long[common].reset_index(drop=True))
    print("identical to shared parquet:", same)
del long

In [ ]:
# 5. Run the full pipeline (development, validation, shift, updating, fairness, attribution, sensitivity)
import run_all
run_all.main()
import export_models   # fitted model parameters without person-level data (TRIPOD+AI item 22)
export_models.main(os.environ["RTW_CACHE_DIR"], os.path.join(os.environ["RTW_OUT_DIR"], "models"))

In [ ]:
# 6. Main tables (aggregated)
OUT = os.environ["RTW_OUT_DIR"]
pd.set_option("display.width", 200, "display.max_columns", 30)
for task in ("A1", "A2", "P"):
    for w in ("unweighted", "weighted"):
        t = pd.read_csv(os.path.join(OUT, "tables", f"perf_C3_{task}_{w}.csv"))
        print(task, w)
        display(t.pivot(index="model", columns="metric", values="estimate").round(3))
display(pd.read_csv(os.path.join(OUT, "tables", "perf_C3_P_within_strata.csv")).query("metric == 'AUROC'").round(3))
display(pd.read_csv(os.path.join(OUT, "tables", "attrition.csv")).round(1))
display(pd.read_csv(os.path.join(OUT, "tables", "shift_adversarial.csv")).round(3))

In [ ]:
# 7. Figures
from IPython.display import Image, display
import glob
for f in sorted(glob.glob(os.path.join(OUT, "figures", "*.png"))):
    print(os.path.basename(f)); display(Image(filename=f, width=800))